# 자금세탁 탐지 AI — 방향 그래프 GNN 엣지 분류 **v7 (일반화 검증판)**

v6 §19 「남은 것 / 권장 다음 단계」 **5개 과제를 전부 구현**한 버전입니다.
목표는 v6과 동일하게 **일반화 성능**이며, v7은 특히 *"이 성능이 다른 세트·다른 분포에서도 유지되는가"*를
측정·감시하는 장치를 갖추는 데 집중했습니다.

### v6 → v7 변경 요약 (남은 과제 대응표)

| # | v6 §19의 남은 과제 | v7에서 한 일 | 위치 |
|---|---|---|---|
| 1 | **세트 간 비교 (화두 7)** — 우선순위 1순위 | §15를 스캐폴드에서 **실제 zero-shot 전이 평가**로 개편. HI-Small로 학습한 앙상블을 LI-Small 등 타 세트에 그대로 적용해 PR-AUC·lift 비교. 피처 파이프라인을 함수화해 세트가 바뀌어도 동일 코드로 처리 | §15 |
| 2 | **POS_WEIGHT_CAP 스윕 (5/15/30)** | 시드 42 고정 스윕 후 **검증 PR-AUC 기준으로 cap 자동 선택** → 본학습(§11)에 반영. test 수치는 참고용으로만 표기(선택에 사용 안 함) | §10-2 |
| 3 | **계좌 단위 보조 판정 (화두 1)** | ① 모델에 **계좌(노드) 보조 헤드** 추가 — 멀티태스크 손실로 "세탁 관여 계좌" 신호를 함께 학습(라벨 노이즈 완화·정규화 효과) ② 추론 후 거래 점수를 **계좌 단위로 집계**해 계좌 레벨 정밀도/재현율 병행 보고 | §9·§10·§12-2 |
| 4 | **드리프트 모니터링** | 검증→테스트 PR-AUC 하락폭(상대 %)에 **주의/경보 임계값** 부여, 점수 분포 PSI, 경보 예산 임계값의 경보율 안정성, 테스트 윈도우별 롤링 PR-AUC → `drift_report.json` 저장 | §16 |
| 5 | **상대 시간차 피처 제거 A/B** | 동일 시드·동일 프로토콜로 **포함(A) vs 제거(B)** 재학습 비교. 하락폭이 크면 합성 생성기 시간 패턴 의존 → 실데이터 이관 시 필수 재검증 항목으로 기록 | §13-2 |

> 시간이 오래 걸리는 실험은 전부 플래그로 제어합니다: `RUN_POS_SWEEP`, `RUN_TIME_AB`, `RUN_TRANSFER`, `RUN_ABLATION`.
> 기본값은 §19 우선순위에 따라 스윕·A/B·전이 평가 **ON**, 전체 ablation **OFF**입니다.

## ⚠️ 화두 결론대로 진행할 때의 위험성 경고 (요청사항 · v7 갱신)

| 화두 결론 | 잠재 위험 | 완화책 (v7 기준) |
|---|---|---|
| **거래 1건씩 판정** (화두1) | 세탁은 여러 거래가 모인 *패턴* 현상 → 개별 거래 라벨은 노이즈가 크고, 시나리오(팬인/팬아웃/사이클) 전체를 놓칠 수 있음 | GNN의 이웃 맥락 집계 + **[v7] 계좌 보조 헤드(멀티태스크)** + **[v7] §12-2 계좌 단위 집계 판정** 으로 3중 완화 |
| **분 단위 배치** (화두2) | 거래 급증 시 배치가 커져 지연 SLA 위반 가능 | §18에서 배치 크기별 지연 측정. 운영 시 배치 크기 상한 + 백프레셔 필요 |
| **사후 탐지** (화두3) | 자금 이탈 후 탐지 → 동결 실효성↓ | 탐지 지연 분포를 모니터링 지표로 관리 권장 |
| **통화별 개별 표준화, 환산 안 함** (화두8) | 희소 통화의 불안정한 z-score, 크로스커런시 패턴 일부 유실 | 희소 통화는 전체 통계로 폴백. 향후 환산 금액 보조 피처 A/B 권장 |
| **로그 변환만, 극단값 방치** (화두9) | 초고액 이상치가 손실을 지배할 수 있음 | z-score가 일부 완화. 불안정 시 클리핑 재논의 |
| **절대시간 제거, 순서만** (화두10·17) | 시간대 신호 손실, 혼잡 구간에서 순서 간격 왜곡 | 상대 시간차 Δt·최근 빈도만 도입(§7-3). **[v7] §13-2 제거 A/B로 의존도를 수치화** |
| **완전 동일 중복 제거** (화두11) | 동일 금액·시각 반복 송금 자체가 세탁 신호일 수 있음 | 제거 건수 로그 출력. 중복 횟수 피처화 재논의 권장 |
| **자기송금 유지** (화두11) | Reinvestment 자기루프 노이즈 | `self_loop` 플래그로 구분 |
| **꼬리 절단** (화두17) | 꼬리 구간 세탁 시나리오 유실 | 절단 전후 시각화·유실 라벨 수 출력 |
| **세트 미통합** (화두7) | 타 은행/타 분포 일반화 검증 불가 | **[v7] §15 zero-shot 전이 평가로 직접 검증** (세트 파일만 있으면 자동) |
| **병렬 엣지 전부 유지** (화두16) | 대형 세트 메모리 병목 | 컨텍스트 윈도우 미니배치로 상한 고정 |
| **상대 시간차 도입** (v6) | 합성 생성기 시간 패턴 과적합 → 실데이터 성능 하락 가능 | **[v7] §13-2 A/B로 기여분 분리 측정 + 이관 체크리스트 기록** |
| **시드 앙상블** (v6) | 모델 4개 서빙 → 지연·운영비 증가 | §12에 단일 모델 성능 병행 보고 |
| **[v7 신규] 계좌 보조 손실** | 보조 라벨(윈도우 내 세탁 관여 계좌)도 거래 라벨에서 유도되므로 같은 노이즈를 일부 공유 | 보조 가중치 `AUX_NODE_W`로 영향 제한. §13 ablation 사다리에 기여분 분리 측정 항목 포함 |
| **[v7 신규] cap 스윕의 선택 편향** | 스윕에서 test 수치를 보고 cap을 고르면 테스트셋 오염 | **선택 기준을 val PR-AUC로 고정** — test 컬럼은 표기만 하고 선택에 사용하지 않음 |

## 4. 환경 설정

In [ ]:
# 최초 1회만 실행하세요 (주석 해제 후):
# %pip install pandas numpy matplotlib networkx scikit-learn torch torch_geometric --break-system-packages

import importlib
for pkg in ["pandas", "numpy", "matplotlib", "networkx", "sklearn", "torch"]:
    try:
        m = importlib.import_module(pkg)
        print(f"[OK] {pkg} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"[없음] {pkg} — 위 설치 셀을 실행하세요")

In [ ]:
import json, os, random, time, warnings, zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score)

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# ----------------------------- 설정 -----------------------------
ZIP_PATH = Path("IBM_AML_dataset/archive (2).zip")
DATASET  = "HI-Small"        # 학습 세트. 화두7 검증은 §15 전이 평가에서 수행
MAX_ROWS = None

TRAIN_FRAC, VAL_FRAC = 0.6, 0.2
TAIL_MIN_FRAC = 0.05

# --- v6 구조 (유지) ---
WINDOW_SIZE   = 100_000
CONTEXT_EDGES = 100_000
USE_TIME_FEATS = True
USE_PORTS      = True
USE_MEGA       = True
NODE_FEAT_DIM  = 9

HIDDEN, N_LAYERS, DROPOUT = 128, 3, 0.15
LR, WEIGHT_DECAY = 2e-3, 1e-5
EPOCHS, PATIENCE = 60, 8
POS_WEIGHT_CAP = 15.0        # §10-2 스윕이 켜져 있으면 검증 성능 기준으로 자동 갱신됨
GRAD_CLIP = 2.0

SEED_LIST = [0, 1, 2, 3]
SEED = SEED_LIST[0]
ALERT_BUDGET = 0.001         # 경보 예산: 전체 거래의 0.1%만 조사 인력에 이관 가정

# --- [v7 신규] §19 남은 과제 대응 설정 ---
RUN_POS_SWEEP = True         # 과제2: POS_WEIGHT_CAP 스윕 (시드 42, 설정당 1회 학습)
POS_CAP_SWEEP = [5.0, 15.0, 30.0]

USE_NODE_AUX = True          # 과제3: 계좌(노드) 보조 헤드 멀티태스크 학습
AUX_NODE_W   = 0.3           #   보조 손실 가중치 (0이면 v6과 동일)

RUN_TIME_AB = True           # 과제5: 상대 시간차 피처 제거 A/B (재학습 1~2회)

RUN_TRANSFER  = True         # 과제1: 타 세트 zero-shot 전이 평가 (파일이 있을 때만)
TRANSFER_SETS = ["LI-Small"] # Medium 세트는 메모리 여유 확인 후 추가 권장 (수천만 건)

DRIFT_WARN, DRIFT_ALERT = 0.10, 0.20   # 과제4: val→test PR-AUC 상대 하락 주의/경보 임계값
PSI_WARN, PSI_ALERT     = 0.10, 0.25   #   점수 분포 PSI 임계값 (업계 관행)

USE_AMP = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {DEVICE}")
print(f"윈도우 = 대상 {WINDOW_SIZE:,} + 과거 컨텍스트 {CONTEXT_EDGES:,}")
print(f"실험 플래그: 스윕={RUN_POS_SWEEP}, 계좌보조={USE_NODE_AUX}(w={AUX_NODE_W}), "
      f"시간A/B={RUN_TIME_AB}, 전이={RUN_TRANSFER} {TRANSFER_SETS}")

In [ ]:
# 한글 폰트 (Linux 환경에서 1회)
import matplotlib.font_manager as fm
_fp = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if Path(_fp).exists():
    fm.fontManager.addfont(_fp)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("✅ 한글 폰트 설정 완료")
else:
    print("나눔고딕 없음 — 필요 시: !apt-get install -y fonts-nanum")

## 5. 데이터 로드 (zip 스트리밍 + 손상 진단 + CSV 폴백)

[v7] 로드를 `load_transactions()` 함수로 분리했습니다 — §15 전이 평가가 같은 코드로 타 세트를 읽습니다.

In [ ]:
# ---------- 데이터 소스 준비: zip 검증 → 실패 시 CSV 폴백 ----------
CSV_SEARCH_DIRS = [Path("IBM_AML_dataset"), Path("."), ZIP_PATH.parent]

zf = None
if ZIP_PATH.exists():
    size_mb = ZIP_PATH.stat().st_size / 1e6
    with open(ZIP_PATH, "rb") as fh:
        magic = fh.read(4)
    if magic[:2] == b"PK":
        try:
            zf = zipfile.ZipFile(ZIP_PATH)
            print(f"[zip 정상] {ZIP_PATH} ({size_mb:,.1f} MB)")
        except zipfile.BadZipFile:
            print(f"[zip 손상] {ZIP_PATH} ({size_mb:,.1f} MB) — 시작은 zip(PK)인데 끝부분이 없습니다.")
            print("  → 업로드/복사가 중간에 끊겨 파일이 잘린 상태입니다. 재업로드가 필요합니다.")
            print("  → 또는 로컬 PC에서 압축을 풀어 필요한 CSV만 IBM_AML_dataset/ 폴더에 올리세요(권장).")
    else:
        print(f"[zip 아님] {ZIP_PATH} ({size_mb:,.1f} MB), 시작 바이트 = {magic!r}")
        print("  → zip이 아닌 파일입니다(다운로드 오류 페이지, LFS 포인터 등일 수 있음). 원본을 다시 받으세요.")
else:
    print(f"[zip 없음] {ZIP_PATH.resolve()}")

def open_member(fname):
    # zip이 정상이면 zip에서, 아니면 폴더들에서 압축 해제된 파일을 찾는다
    if zf is not None:
        names = set(zf.namelist())
        if fname in names:
            return zf.open(fname)
        inner = [n for n in names if n.endswith("/" + fname)]
        if inner:
            return zf.open(inner[0])
    for d in CSV_SEARCH_DIRS:
        p = Path(d) / fname
        if p.exists():
            print(f"[파일 사용] {p}")
            return open(p, "rb")
    raise FileNotFoundError(
        f"'{fname}' 를 찾지 못했습니다. zip을 온전히 재업로드하거나, "
        f"압축 해제한 파일을 {[str(Path(d)) for d in CSV_SEARCH_DIRS]} 중 한 곳에 두세요."
    )


def load_transactions(ds, nrows=MAX_ROWS):
    '''[v7] 세트 이름으로 거래 CSV를 로드 — 본학습(§5)과 전이 평가(§15)가 공유.'''
    with open_member(f"{ds}_Trans.csv") as f:
        d = pd.read_csv(
            f,
            dtype={
                "From Bank": str, "Account": str, "To Bank": str, "Account.1": str,
                "Receiving Currency": "category", "Payment Currency": "category",
                "Payment Format": "category", "Is Laundering": np.int8,
            },
            nrows=nrows,
        )
    d["Timestamp"] = pd.to_datetime(d["Timestamp"], format="%Y/%m/%d %H:%M")
    return d


t0 = time.time()
df = load_transactions(DATASET)
print(f"{len(df):,}건 로드 ({time.time()-t0:.1f}s)")
df.head()

## 6. 전처리 ① — 중복 제거·정렬·꼬리 절단 (화두 10·11·17)

[v7] `clean_frame()` 함수로 분리 — §15 전이 평가가 타 세트에 동일 규칙을 적용합니다.

In [ ]:
def clean_frame(raw, verbose=True):
    '''중복 제거 → 시각 정렬 → 꼬리 절단. (정제된 df, 메타) 반환. v6 §6과 동일 규칙.'''
    n_raw = len(raw)
    d = raw.drop_duplicates()
    n_dup = n_raw - len(d)
    d = d.sort_values("Timestamp", kind="mergesort").reset_index(drop=True)

    daily = d["Timestamp"].dt.floor("D").value_counts().sort_index()
    thr_ = daily.median() * TAIL_MIN_FRAC
    good_days = daily[daily >= thr_].index
    ms, me = good_days.min(), good_days.max()

    day = d["Timestamp"].dt.floor("D")
    mask_main = (day >= ms) & (day <= me)
    lost_pos = int(d.loc[~mask_main, "Is Laundering"].sum())
    if verbose:
        print(f"중복 제거: {n_dup:,}건 삭제 ({n_dup/max(n_raw,1)*100:.3f}%)")
        print(f"주 기간: {ms.date()} ~ {me.date()}  "
              f"(절단 {int((~mask_main).sum()):,}건, 그중 세탁 라벨 {lost_pos}건 유실 — 경고 섹션 참조)")
    d = d[mask_main].reset_index(drop=True)
    if verbose:
        print(f"최종 {len(d):,}건, 세탁 비율 {d['Is Laundering'].mean()*100:.4f}%")
    return d, dict(daily_counts_all=daily, thr=thr_, main_start=ms, main_end=me,
                   lost_pos=lost_pos, n_dup=n_dup)


df, CLEAN_META = clean_frame(df)
daily_counts_all, thr = CLEAN_META["daily_counts_all"], CLEAN_META["thr"]
main_start, main_end = CLEAN_META["main_start"], CLEAN_META["main_end"]
n = len(df)

## 7. 전처리 ② — 피처 엔지니어링

[v7] 피처 생성을 `add_base_features()` / `add_time_features()` 함수로 분리했습니다.
수식·순서는 v6과 동일하며, §15 전이 평가는 **학습 세트의 범주 어휘(pf/pc/rc)** 를 그대로 넘겨
입력 차원을 일치시킵니다(통화 정규화 통계만 타깃 세트의 train 구간으로 적합 — 모델 학습에는 사용되지 않음).

In [ ]:
df["order_idx"] = np.arange(n, dtype=np.int64)

n_tr = int(n * TRAIN_FRAC)
n_va = int(n * (TRAIN_FRAC + VAL_FRAC))
split = np.full(n, "test", dtype=object)
split[:n_tr] = "train"; split[n_tr:n_va] = "val"
df["split"] = split
is_train = df["split"].eq("train").to_numpy()
print(f"train {n_tr:,} / val {n_va-n_tr:,} / test {n-n_va:,}")

src_key = df["From Bank"].astype(str) + "_" + df["Account"].astype(str)
dst_key = df["To Bank"].astype(str) + "_" + df["Account.1"].astype(str)
codes, uniques = pd.factorize(pd.concat([src_key, dst_key], ignore_index=True))
df["src_id"] = codes[:n].astype(np.int64)
df["dst_id"] = codes[n:].astype(np.int64)
NUM_ACCOUNTS = len(uniques)
print(f"계좌(노드) 수: {NUM_ACCOUNTS:,}")

df["self_loop"] = (df["src_id"] == df["dst_id"]).astype(np.float32)
print(f"자기송금: {int(df['self_loop'].sum()):,}건 유지")

In [ ]:
def add_base_features(d, is_tr, cats=None):
    '''기본 엣지 피처 (v6 §7과 동일 수식).

    is_tr : 통계 적합에 쓸 train 마스크 (누수 차단).
    cats  : (pf, pc, rc) 범주 어휘. None이면 train에서 적합, 주어지면(전이 평가) 그대로 사용해
            입력 차원을 학습 세트와 일치시킨다.
    반환   : (피처가 추가된 d, 기본 피처 이름 목록, (pf, pc, rc) 어휘)
    '''
    def per_currency_z(amount_col, currency_col):
        # (화두 8·9) 통화별 log1p → z-score. train 통계만 사용, 희소/미등장 통화는 전체 통계로 폴백.
        la = np.log1p(d[amount_col].clip(lower=0).to_numpy(dtype=np.float64))
        cur = d[currency_col].astype(str)
        s = pd.DataFrame({"cur": cur, "la": la})
        st = s[is_tr].groupby("cur")["la"].agg(["mean", "std"])
        st["std"] = st["std"].replace(0, np.nan)
        g_mean, g_std = s.loc[is_tr, "la"].mean(), s.loc[is_tr, "la"].std()
        m_ = cur.map(st["mean"]).astype(np.float64).fillna(g_mean).to_numpy()
        sd = cur.map(st["std"]).astype(np.float64).fillna(g_std).to_numpy()
        return ((la - m_) / sd).astype(np.float32)

    d["amt_paid_z"] = per_currency_z("Amount Paid", "Payment Currency")
    d["amt_recv_z"] = per_currency_z("Amount Received", "Receiving Currency")
    d["cur_mismatch"] = (d["Payment Currency"].astype(str) != d["Receiving Currency"].astype(str)).astype(np.float32)
    d["amt_neq"] = (d["Amount Paid"] != d["Amount Received"]).astype(np.float32)

    def onehot(col, prefix, fixed, max_cats=20):
        cs = fixed if fixed is not None else d.loc[is_tr, col].astype(str).value_counts().index[:max_cats].tolist()
        v = d[col].astype(str)
        return pd.DataFrame({f"{prefix}_{c}": (v == c).astype(np.float32) for c in cs}, index=d.index), list(cs)

    pf_oh, pf_c = onehot("Payment Format", "pf", cats[0] if cats else None)
    pc_oh, pc_c = onehot("Payment Currency", "pc", cats[1] if cats else None)
    rc_oh, rc_c = onehot("Receiving Currency", "rc", cats[2] if cats else None)
    d = pd.concat([d, pf_oh, pc_oh, rc_oh], axis=1)

    # (화두 10) 순서 기반 시퀀스/간격 — 과거 방향만 사용
    nloc = len(d)
    logN = np.log1p(nloc)
    for role, key in [("src", "src_id"), ("dst", "dst_id")]:
        g = d.groupby(key)["order_idx"]
        d[f"{role}_seq"] = (np.log1p(g.cumcount()) / logN).astype(np.float32)
        prev = g.shift(1)
        d[f"{role}_first"] = prev.isna().astype(np.float32)
        d[f"{role}_gap"] = (np.log1p((d["order_idx"] - prev).fillna(nloc)) / logN).astype(np.float32)

    base = (["amt_paid_z", "amt_recv_z", "cur_mismatch", "amt_neq", "self_loop",
             "src_seq", "src_first", "src_gap", "dst_seq", "dst_first", "dst_gap"]
            + list(pf_oh.columns) + list(pc_oh.columns) + list(rc_oh.columns))
    return d, base, (pf_c, pc_c, rc_c)


df, BASE_FEATURES, (PF_CATS, PC_CATS, RC_CATS) = add_base_features(df, is_train)
print(f"기본 엣지 피처 {len(BASE_FEATURES)}차원")

### 7-3. [v6 · 제안 ③] 상대 시간차 피처 — 화두 10 재논의 반영

화두 10 결론은 "절대시간 제거"였고, v5는 **순서 인덱스 간격**만 썼습니다. v6부터 **절대시각(시·요일·날짜)은
여전히 쓰지 않고** 계좌·계좌쌍별 Δt와 최근 1h/24h 빈도만 도입합니다.

| 피처 | 의미 | AML 근거 |
|---|---|---|
| `src_dt`, `dst_dt` | 해당 계좌의 **직전 거래로부터 경과 시간** (log 정규화) | 휴면 계좌의 갑작스러운 활성화 |
| `pair_dt`, `pair_first` | **같은 (송→수) 쌍**의 직전 거래 경과 시간 / 첫 거래 여부 | 신규 상대방으로의 송금 |
| `out_cnt*`, `in_cnt*` | 최근 1시간·24시간 내 **보낸/받은 건수** | 팬아웃·팬인 속도 |
| `src_cnt*`, `dst_cnt*` | 역할 무관 최근 활동량 | 계좌 회전율 급증 |
| `pair_cnt*` | 같은 쌍의 최근 반복 횟수 | 스머핑(분할 반복 송금) |

**누수 차단:** 모든 값은 `searchsorted(..., side="left")` 로 자기 자신·동시각 거래를 제외한 **과거만** 집계.
**[v7]** 이 피처군은 합성 생성기 패턴에 과적합됐을 가능성이 있어 **§13-2에서 제거 A/B를 수행**합니다.

In [ ]:
def prior_stats(acct, tsec, windows):
    '''계좌(또는 계좌쌍)별 ① 직전 이벤트까지의 Δt ② 과거 W초 내 건수. 과거만 사용(누수 없음).'''
    K = np.int64(10_000_000)                       # 계좌 블록 분리용 (tsec < K 보장)
    keys = acct.astype(np.int64) * K + tsec
    skeys = np.sort(keys, kind="stable")
    pos = np.searchsorted(skeys, keys, side="left")            # 자기·동시각 제외
    prev_idx = pos - 1
    ok = prev_idx >= 0
    prev_key = np.where(ok, skeys[np.clip(prev_idx, 0, None)], -1)
    same = ok & ((prev_key // K) == acct.astype(np.int64))
    dt = np.where(same, tsec - (prev_key % K), -1).astype(np.float64)
    out = {"dt": dt, "is_first": (~same).astype(np.float32)}
    for w in windows:
        lo = np.searchsorted(skeys, acct.astype(np.int64) * K + np.maximum(tsec - w, 0), side="left")
        out[f"cnt{w}"] = (pos - lo).astype(np.float32)
    return out


def add_time_features(d):
    '''[v7] 상대 시간차 피처 (v6 §7-3과 동일 수식). (d, 피처 이름 목록) 반환.'''
    if not USE_TIME_FEATS:
        return d, []
    nloc = len(d)
    tfeats = []
    ts_abs = d["Timestamp"].to_numpy().astype("datetime64[s]").astype(np.int64)
    tsec = ts_abs - ts_abs.min()          # 상대 초. 절대시각 자체는 피처로 쓰지 않음
    assert tsec.max() < 10_000_000
    LOGT = np.log1p(86400.0 * 14)         # 정규화 상수 (2주)
    LOGC = np.log(50.0)
    WIN = [3600, 86400]
    src_np, dst_np = d["src_id"].to_numpy(), d["dst_id"].to_numpy()
    pair_id = pd.factorize(pd.Series(src_np * np.int64(1 << 21) + dst_np))[0].astype(np.int64)

    both = prior_stats(np.concatenate([src_np, dst_np]), np.concatenate([tsec, tsec]), WIN)
    for role, sl in [("src", slice(0, nloc)), ("dst", slice(nloc, 2 * nloc))]:
        dd = both["dt"][sl]
        d[f"{role}_dt"] = np.where(dd < 0, 1.0, np.log1p(np.maximum(dd, 0)) / LOGT).astype(np.float32)
        d[f"{role}_dt_first"] = both["is_first"][sl]
        tfeats += [f"{role}_dt", f"{role}_dt_first"]
        for w in WIN:
            d[f"{role}_cnt{w}"] = (np.log1p(both[f"cnt{w}"][sl]) / LOGC).astype(np.float32)
            tfeats.append(f"{role}_cnt{w}")

    out_s, in_s = prior_stats(src_np, tsec, WIN), prior_stats(dst_np, tsec, WIN)
    for w in WIN:
        d[f"out_cnt{w}"] = (np.log1p(out_s[f"cnt{w}"]) / LOGC).astype(np.float32)
        d[f"in_cnt{w}"] = (np.log1p(in_s[f"cnt{w}"]) / LOGC).astype(np.float32)
        tfeats += [f"out_cnt{w}", f"in_cnt{w}"]

    pr = prior_stats(pair_id, tsec, WIN)
    d["pair_dt"] = np.where(pr["dt"] < 0, 1.0, np.log1p(np.maximum(pr["dt"], 0)) / LOGT).astype(np.float32)
    d["pair_first"] = pr["is_first"]
    tfeats += ["pair_dt", "pair_first"]
    for w in WIN:
        d[f"pair_cnt{w}"] = (np.log1p(pr[f"cnt{w}"]) / LOGC).astype(np.float32)
        tfeats.append(f"pair_cnt{w}")
    return d, tfeats


df, TIME_FEATURES = add_time_features(df)
EDGE_FEATURES = BASE_FEATURES + TIME_FEATURES
print(f"상대 시간차 피처 {len(TIME_FEATURES)}종 추가 → 엣지 피처 총 {len(EDGE_FEATURES)}차원")

## 8. 탐색적 데이터 분석(EDA) 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 10))

# (1) 클래스 불균형
ax = axes[0, 0]
vc = df["Is Laundering"].value_counts()
bars = ax.bar(["정상 (0)", "세탁 (1)"], vc.reindex([0, 1]).values, color=["#4C72B0", "#C44E52"])
ax.set_yscale("log"); ax.set_title(f"클래스 불균형 (세탁 {df['Is Laundering'].mean()*100:.4f}%)")
for b, v in zip(bars, vc.reindex([0, 1]).values):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:,}", ha="center", va="bottom")

# (2) 상위 통화별 금액 분포 (log1p) — 화두 8 근거: 통화별 스케일 상이
ax = axes[0, 1]
top_cur = df["Payment Currency"].astype(str).value_counts().index[:6]
data = [np.log1p(df.loc[df["Payment Currency"].astype(str) == c, "Amount Paid"].clip(lower=0)) for c in top_cur]
ax.boxplot(data, showfliers=False)
ax.set_xticklabels(list(top_cur), rotation=30)
ax.set_title("통화별 log1p(지불금액) 분포 — 통화별 표준화 근거")

# (3) 결제수단별 세탁 비율
ax = axes[0, 2]
rate = df.groupby(df["Payment Format"].astype(str))["Is Laundering"].mean().sort_values() * 100
ax.barh(rate.index, rate.values, color="#C44E52")
ax.set_title("결제수단별 세탁 비율(%)"); ax.set_xlabel("%")

# (4) 일별 거래량 + 주 기간(꼬리 절단) 경계 — 화두 17
ax = axes[1, 0]
ax.plot(daily_counts_all.index, daily_counts_all.values, marker="o", ms=3, color="#4C72B0")
ax.axvspan(main_start, main_end, color="green", alpha=0.10, label="주 기간(학습 사용)")
ax.axhline(thr, color="red", ls="--", lw=1, label=f"절단 임계(중앙값×{TAIL_MIN_FRAC}")
ax.set_yscale("log"); ax.legend(); ax.set_title("일별 거래량과 꼬리 절단 (화두 17)")
ax.tick_params(axis="x", rotation=30)

# (5) 계좌 차수(degree) 분포 (log-log) — 그래프 접근의 근거
ax = axes[1, 1]
deg = np.bincount(np.concatenate([df["src_id"].to_numpy(), df["dst_id"].to_numpy()]))
dv, dc = np.unique(deg[deg > 0], return_counts=True)
ax.scatter(dv, dc, s=6, alpha=0.5, color="#55A868")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("계좌 차수"); ax.set_ylabel("계좌 수"); ax.set_title("계좌 차수 분포 (멱법칙형)")

# (6) 순서 축에서의 세탁 비율 + 분할 경계 — 화두 10
ax = axes[1, 2]
bins = 60
bin_idx = (df["order_idx"] // max(1, n // bins)).clip(upper=bins - 1)
br = df.groupby(bin_idx)["Is Laundering"].mean() * 100
ax.plot(br.index, br.values, color="#8172B2")
for b, lab in [(n_tr / (n / bins), "train|val"), (n_va / (n / bins), "val|test")]:
    ax.axvline(b, color="k", ls=":")
    ax.text(b, ax.get_ylim()[1]*0.9 if ax.get_ylim()[1] else 0.1, lab, rotation=90, va="top", fontsize=8)
ax.set_title("순서 구간별 세탁 비율(%)과 분할 경계"); ax.set_xlabel("순서 구간(bin)")

plt.tight_layout(); plt.show()

### 8-2. 상대 시간차 피처의 판별력 확인

In [ ]:
if USE_TIME_FEATS:
    show = ["pair_dt", "pair_first", "out_cnt3600", "in_cnt3600", "src_dt", "pair_cnt86400"]
    label = {"pair_dt": "같은 쌍 직전거래 Δt", "pair_first": "해당 쌍 첫 거래 비율",
             "out_cnt3600": "최근 1h 보낸 건수", "in_cnt3600": "최근 1h 받은 건수",
             "src_dt": "보내는 계좌 직전거래 Δt", "pair_cnt86400": "같은 쌍 최근 24h 반복"}
    y_all = df["Is Laundering"].to_numpy()
    fig, axes = plt.subplots(2, 3, figsize=(17, 8))
    for ax, c in zip(axes.ravel(), show):
        v0, v1 = df.loc[y_all == 0, c], df.loc[y_all == 1, c]
        bins = np.linspace(float(df[c].min()), float(df[c].max()), 50)
        ax.hist(v0, bins=bins, density=True, alpha=0.55, color="#4C72B0", label="정상")
        ax.hist(v1, bins=bins, density=True, alpha=0.65, color="#C44E52", label="세탁")
        ax.set_title(f"{label[c]}\n정상 {v0.mean():.3f} vs 세탁 {v1.mean():.3f}", fontsize=10)
        ax.legend(fontsize=8)
    plt.suptitle("상대 시간차 피처 — 세탁/정상 분포 비교 (분리도가 클수록 유용)", fontsize=13)
    plt.tight_layout(); plt.show()

    # 단변량 판별력 (AUC-ROC 대용: 순위 상관)
    from sklearn.metrics import roc_auc_score
    rank = sorted(((roc_auc_score(y_all, df[c].to_numpy()), c) for c in TIME_FEATURES),
                  key=lambda t: -abs(t[0] - 0.5))
    print("단변량 ROC-AUC (0.5에서 멀수록 판별력 있음)")
    for a, c in rank[:8]:
        print(f"  {c:<16} {a:.4f}")

### 8-3. 실제 세탁 패턴 서브그래프 시각화 (`*_Patterns.txt`)

In [ ]:
with open_member(f"{DATASET}_Patterns.txt") as f:
    pat_lines = f.read().decode("utf-8", errors="ignore").splitlines()

pattern_blocks, cur = [], None
for ln in pat_lines:
    if ln.startswith("BEGIN LAUNDERING ATTEMPT"):
        cur = {"name": ln.split("-", 1)[-1].strip(), "rows": []}
    elif ln.startswith("END LAUNDERING ATTEMPT"):
        if cur and cur["rows"]:
            pattern_blocks.append(cur)
        cur = None
    elif cur is not None and ln.strip():
        parts = ln.split(",")
        if len(parts) >= 11:
            cur["rows"].append(parts)
print(f"세탁 시나리오 블록 {len(pattern_blocks):,}개 파싱")

# 서로 다른 유형 3개 골라 시각화
seen_types, picks = set(), []
for b in pattern_blocks:
    t = b["name"].split(":")[0].strip()
    if t not in seen_types and 3 <= len(b["rows"]) <= 40:
        seen_types.add(t); picks.append(b)
    if len(picks) == 3:
        break

fig, axes = plt.subplots(1, len(picks), figsize=(6.5 * len(picks), 5.5))
axes = np.atleast_1d(axes)
for ax, blk in zip(axes, picks):
    G = nx.MultiDiGraph()
    for r in blk["rows"]:
        u = f"{r[1]}_{r[2][-4:]}"; v = f"{r[3]}_{r[4][-4:]}"
        G.add_edge(u, v)
    pos = nx.spring_layout(G, seed=SEED, k=1.2)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=420, node_color="#F4A5A5", edgecolors="#C44E52")
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#C44E52", arrows=True,
                           connectionstyle="arc3,rad=0.08", alpha=0.8)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=6)
    ax.set_title(f"세탁 패턴: {blk['name']}  ({len(blk['rows'])}건)")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 9. 그래프 구성과 모델

### 9-1. 컨텍스트 겹침 윈도우 (v6 유지)

윈도우마다 앞쪽에 `CONTEXT_EDGES` 건을 그래프 컨텍스트로 붙이고, **분류·손실은 대상 구간에만** 적용합니다.

```
v6/v7: [ctx][==대상1==]
            [ctx      ][==대상2==]     ← 모든 대상 거래가 과거 맥락을 가짐 (학습-서빙 정합)
```

**[v7 신규] 데이터 번들과 계좌 보조 라벨**

* 그래프 빌더가 전역 배열 대신 **번들(dict)** 을 받도록 일반화 — §15 전이 평가가 타 세트 번들을 같은 코드로 처리합니다.
* 각 윈도우에서 **대상 구간의 세탁 거래에 관여한 계좌**를 1로 하는 노드 라벨(`node_y`)과, 대상 구간에
  등장한 노드 마스크(`node_mask`)를 함께 만듭니다 → §10의 **계좌 보조 손실**(화두 1 대응)에 사용.
  라벨은 대상 구간에서만 유도하므로 v6과 동일하게 **미래 정보 누수가 없습니다.**

In [ ]:
SRC = df["src_id"].to_numpy()
DST = df["dst_id"].to_numpy()
Y   = df["Is Laundering"].to_numpy().astype(np.float32)
EF  = np.ascontiguousarray(df[EDGE_FEATURES].to_numpy(dtype=np.float32))
A_PAID = df["amt_paid_z"].to_numpy()
A_RECV = df["amt_recv_z"].to_numpy()

EDGE_EXTRA = 2 + (2 if USE_PORTS else 0)      # 방향 플래그 2 + 포트(순번, 다중도) 2
EDGE_DIM   = len(EDGE_FEATURES) + EDGE_EXTRA

# [v7] 본학습 데이터 번들 — §15 전이 평가는 타 세트로 같은 구조를 만든다
MAIN = dict(SRC=SRC, DST=DST, Y=Y, EF=EF, AP=A_PAID, AR=A_RECV,
            n=n, n_tr=n_tr, n_va=n_va, cache={})


def build_window(ctx_lo, lo, hi, D=None):
    '''[ctx_lo, hi) 로 방향 멀티그래프를 만들고, [lo, hi) 만 분류 대상으로 표시.'''
    D = MAIN if D is None else D
    m = hi - ctx_lo                                    # 그래프에 들어가는 원본 엣지 수
    s, d = D["SRC"][ctx_lo:hi], D["DST"][ctx_lo:hi]
    nodes, inv = np.unique(np.concatenate([s, d]), return_inverse=True)
    sl, dl = inv[:m].astype(np.int64), inv[m:].astype(np.int64)
    nn_ = len(nodes)

    out_deg = np.bincount(sl, minlength=nn_).astype(np.float32)
    in_deg  = np.bincount(dl, minlength=nn_).astype(np.float32)
    ap, ar  = D["AP"][ctx_lo:hi], D["AR"][ctx_lo:hi]
    mean_out = (np.bincount(sl, weights=ap, minlength=nn_) / np.maximum(out_deg, 1)).astype(np.float32)
    mean_in  = (np.bincount(dl, weights=ar, minlength=nn_) / np.maximum(in_deg, 1)).astype(np.float32)

    pair_f = sl * nn_ + dl                             # 윈도우 내 (u,v) 쌍 코드
    cols = [np.log1p(out_deg), np.log1p(in_deg), mean_out, mean_in]
    if NODE_FEAT_DIM > 4:
        uf = np.unique(pair_f)
        n_out_nb = np.bincount(uf // nn_, minlength=nn_).astype(np.float32)   # 고유 수신 상대방 수
        n_in_nb  = np.bincount(uf % nn_,  minlength=nn_).astype(np.float32)   # 고유 송신 상대방 수
        deg = out_deg + in_deg
        cols += [np.log1p(n_out_nb), np.log1p(n_in_nb),
                 ((out_deg - in_deg) / np.maximum(deg, 1)).astype(np.float32),
                 (out_deg / np.maximum(n_out_nb, 1)).astype(np.float32),
                 (in_deg / np.maximum(n_in_nb, 1)).astype(np.float32)]
    x = np.stack(cols, axis=1).astype(np.float32)

    ei = np.stack([sl, dl])
    ei_full = np.concatenate([ei, ei[::-1]], axis=1)   # 역방향 메시지 전달용
    ea = D["EF"][ctx_lo:hi]
    extra = np.zeros((2 * m, EDGE_EXTRA), dtype=np.float32)
    extra[:m, 0] = 1.0; extra[m:, 1] = 1.0             # [1,0]=정방향, [0,1]=역방향
    if USE_PORTS:
        o = np.argsort(pair_f, kind="stable")
        srt = pair_f[o]
        newg = np.r_[True, srt[1:] != srt[:-1]]
        gid = np.cumsum(newg) - 1
        start = np.flatnonzero(newg)
        rank = np.empty(m, np.int64); rank[o] = np.arange(m) - start[gid]
        mult_srt = np.bincount(gid, minlength=len(start)).astype(np.float32)[gid]
        mult = np.empty(m, np.float32); mult[o] = mult_srt
        pn = (np.log1p(rank) / np.log(20)).astype(np.float32)
        mv = (np.log1p(mult) / np.log(20)).astype(np.float32)
        extra[:m, 2] = pn; extra[m:, 2] = pn
        extra[:m, 3] = mv; extra[m:, 3] = mv
    ea_full = np.concatenate([np.concatenate([ea, ea], axis=0), extra], axis=1)

    # 병렬 엣지 그룹 (MEGA 2단계 집계용) — 양방향 전체
    pf_full = np.concatenate([pair_f, dl * nn_ + sl])
    pu, pinv = np.unique(pf_full, return_inverse=True)
    pair_dst = (pu % nn_).astype(np.int64)

    # [v7] 계좌 보조 라벨 — 대상 구간의 세탁 거래에 관여한 계좌 (과거/미래 누수 없음)
    tgt = slice(lo - ctx_lo, hi - ctx_lo)
    sl_t, dl_t = sl[tgt], dl[tgt]
    y_t = D["Y"][lo:hi]
    node_y = np.zeros(nn_, np.float32)
    pos_m = y_t > 0
    if pos_m.any():
        node_y[sl_t[pos_m]] = 1.0
        node_y[dl_t[pos_m]] = 1.0
    node_mask = np.zeros(nn_, bool)
    node_mask[sl_t] = True; node_mask[dl_t] = True

    return dict(x=torch.from_numpy(x), ei=torch.from_numpy(ei_full), ea=torch.from_numpy(ea_full),
                y=torch.from_numpy(y_t), m=m,
                pinv=torch.from_numpy(pinv.astype(np.int64)),
                pair_dst=torch.from_numpy(pair_dst), n_pairs=len(pu),
                node_y=torch.from_numpy(node_y), node_mask=torch.from_numpy(node_mask),
                tgt=tgt)


def window_ranges(lo, hi, size=None, ctx=None):
    '''(ctx_lo, lo, hi) 삼중항 목록. ctx_lo 는 구간 시작 이전 ctx 건.'''
    size = size or WINDOW_SIZE
    ctx = CONTEXT_EDGES if ctx is None else ctx
    return [(max(a - ctx, 0), a, min(a + size, hi)) for a in range(lo, hi, size)]


TRAIN_W = window_ranges(0, n_tr)
VAL_W   = window_ranges(n_tr, n_va)
TEST_W  = window_ranges(n_va, n)

# 윈도우 구성(np.unique/정렬)은 CPU에서 창당 ~0.3초 → 반복 학습 시 낭비가 큽니다.
# 창 구성 결과는 시드와 무관하므로 본학습 번들만 CPU 메모리에 캐시합니다.
CACHE_WINDOWS = True
def get_window(w, D=None):
    D = MAIN if D is None else D
    if not CACHE_WINDOWS:
        return build_window(*w, D=D)
    if w not in D["cache"]:
        D["cache"][w] = build_window(*w, D=D)
    return D["cache"][w]

print(f"윈도우: train {len(TRAIN_W)} / val {len(VAL_W)} / test {len(TEST_W)}")
print(f"엣지 피처 {EDGE_DIM}차원 (기본 {len(BASE_FEATURES)} + 시간 {len(TIME_FEATURES)} + 구조 {EDGE_EXTRA})")
_g = build_window(*TRAIN_W[1])
print(f"예시 윈도우: 노드 {_g['x'].shape[0]:,} / 방향엣지 {_g['ei'].shape[1]:,} / 병렬쌍 {_g['n_pairs']:,} "
      f"/ 분류 대상 {_g['tgt'].stop - _g['tgt'].start:,} / 대상 관여 계좌 {int(_g['node_mask'].sum()):,}"
      f" (세탁 관여 {int(_g['node_y'].sum()):,})")
del _g

### 9-2. MEGA-GNN식 다중엣지 집계 (v6 유지) + [v7] 계좌 보조 헤드

MEGA-GNN(Multi-Edge Aggregation)은 집계를 두 단계로 나눕니다.

1. **① 계좌쌍 내부 집계** — 같은 (u→v) 병렬 엣지들끼리 먼저 합쳐 "이 상대와의 관계" 벡터를 만들고
2. **② 이웃 집계** — 그 관계 벡터들을 노드로 모읍니다.

$$h_v' = \mathrm{MLP}\Big((1+\epsilon)\,h_v + \sum_{u \in \mathcal N(v)} \underbrace{\textstyle\sum_{e \in E_{uv}} \phi(h_u, e)}_{\text{① 병렬 엣지}}\Big)$$

**[v7] 계좌 보조 헤드:** 최종 노드 표현 $h_v$ 위에 작은 MLP를 얹어 *"이 계좌가 이번 윈도우의 세탁 거래에
관여했는가"*를 함께 예측합니다. 화두 1의 근본 위험(개별 거래 라벨 노이즈)에 대한 대응으로,

* 거래 라벨 하나하나는 노이즈가 커도 **계좌 단위로 뭉치면 신호가 안정적**이고,
* 보조 손실이 노드 표현을 "세탁 관여 계좌" 방향으로 정렬해 **엣지 분류의 일반화에도 도움**을 줍니다
  (멀티태스크 정규화). 추론 시 이 헤드는 사용하지 않으며(§12-2는 거래 점수의 계좌 집계), 파라미터 증가는 미미합니다.

In [ ]:
class MegaConv(nn.Module):
    '''GINe + (옵션) 병렬 엣지 2단계 집계.

    집계(index_add_)는 항상 fp32로 수행 — V100에서 fp16 atomicAdd 가 2배 이상 느립니다.
    '''
    def __init__(self, hidden, mega=True):
        super().__init__()
        self.mega = mega
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))

    def forward(self, h, ei, e, pinv, pair_dst, n_pairs):
        msg = F.relu(h[ei[0]] + e)
        m32 = msg.float()
        if self.mega:
            pair = torch.zeros(n_pairs, msg.size(1), device=msg.device)
            pair.index_add_(0, pinv, m32)            # ① 같은 계좌쌍의 병렬 엣지끼리
            agg = torch.zeros(h.size(0), msg.size(1), device=msg.device)
            agg.index_add_(0, pair_dst, pair)        # ② 이웃(계좌쌍) 단위로
        else:
            agg = torch.zeros(h.size(0), msg.size(1), device=msg.device)
            agg.index_add_(0, ei[1], m32)            # 기존 GINe: 한 번에 전부
        return self.mlp((1 + self.eps) * h + agg.to(msg.dtype))


class EdgeGINe(nn.Module):
    '''MEGA 집계 + 엣지 업데이트 + 역방향 메시지 전달 + [v7] 계좌 보조 헤드.'''
    def __init__(self, node_dim, edge_dim, hidden=128, layers=3, dropout=0.15, mega=True):
        super().__init__()
        self.node_in = nn.Linear(node_dim, hidden)
        self.edge_in = nn.Linear(edge_dim, hidden)
        self.convs = nn.ModuleList([MegaConv(hidden, mega) for _ in range(layers)])
        self.eupds = nn.ModuleList([nn.Sequential(nn.Linear(3 * hidden, hidden), nn.ReLU(),
                                                  nn.Linear(hidden, hidden)) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.dropout = dropout
        self.head = nn.Sequential(nn.Linear(3 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.node_head = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(),
                                       nn.Linear(hidden // 2, 1))          # [v7] 계좌 보조

    def forward(self, g):
        h, ei = self.node_in(g["x"]), g["ei"]
        e = self.edge_in(g["ea"])
        for conv, eupd, norm in zip(self.convs, self.eupds, self.norms):
            h = F.relu(norm(conv(h, ei, e, g["pinv"], g["pair_dst"], g["n_pairs"])))
            h = F.dropout(h, self.dropout, self.training)
            e = e + eupd(torch.cat([h[ei[0]], h[ei[1]], e], dim=1))     # 엣지 업데이트
        m, tgt = g["m"], g["tgt"]
        fi = ei[:, :m][:, tgt]                       # 정방향 엣지 중 "대상 구간"만 분류
        z = torch.cat([h[fi[0]], h[fi[1]], e[:m][tgt]], dim=1)
        return self.head(z).squeeze(-1), self.node_head(h).squeeze(-1)


_m = EdgeGINe(NODE_FEAT_DIM, EDGE_DIM, HIDDEN, N_LAYERS, DROPOUT, USE_MEGA)
print(f"파라미터 수: {sum(p.numel() for p in _m.parameters()):,} "
      f"(계좌 보조 헤드 {sum(p.numel() for p in _m.node_head.parameters()):,} 포함)")
del _m

## 10. 학습 — 시드별 재현 가능한 학습 함수

* 손실: `BCEWithLogitsLoss(pos_weight)` (상한 `POS_WEIGHT_CAP` — **§10-2 스윕으로 결정**)
* **[v7] 계좌 보조 손실**: `loss = 엣지 BCE + AUX_NODE_W × 노드 BCE` — 노드 라벨은 윈도우 대상 구간에서만
  유도(§9-1), 마스크된 노드(대상 구간 등장 계좌)에만 적용. `pos_weight`는 윈도우별로 산출하고 같은 cap을 공유
* 모델 선택: **검증 PR-AUC** 조기종료 (일반화 우선; 보조 손실은 선택 기준에 불포함)
* 혼합정밀(fp16) + 집계 fp32

In [ ]:
def to_dev(g):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in g.items()}


def train_one_seed(seed, epochs=EPOCHS, patience=PATIENCE, verbose=True, pos_cap=None, aux_w=None):
    '''[v7] pos_cap: POS_WEIGHT_CAP 오버라이드(§10-2 스윕용) / aux_w: 계좌 보조 손실 가중치 오버라이드.'''
    cap = float(POS_WEIGHT_CAP if pos_cap is None else pos_cap)
    aw = float((AUX_NODE_W if USE_NODE_AUX else 0.0) if aux_w is None else aux_w)

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    model = EdgeGINe(NODE_FEAT_DIM, EDGE_DIM, HIDDEN, N_LAYERS, DROPOUT, USE_MEGA).to(DEVICE)
    pos = float(Y[:n_tr].sum())
    pw = torch.tensor(min((n_tr - pos) / max(pos, 1.0), cap), device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5,
                                                           patience=3, min_lr=1e-5)
    amp_on = USE_AMP and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_on)

    @torch.no_grad()
    def predict(windows):
        model.eval(); ps = []
        for w in windows:
            g = to_dev(get_window(w))
            with torch.autocast("cuda", torch.float16, enabled=amp_on):
                logit, _ = model(g)
            ps.append(torch.sigmoid(logit.float()).cpu().numpy())
            del g, logit
        return np.concatenate(ps)

    hist = {"train_loss": [], "val_ap": []}
    best_ap, best_state, left = -1.0, None, patience
    vy = Y[n_tr:n_va]
    for epoch in range(1, epochs + 1):
        model.train(); ep_loss = 0.0; t0 = time.time()
        for wi in np.random.permutation(len(TRAIN_W)):
            g = to_dev(get_window(TRAIN_W[wi]))
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", torch.float16, enabled=amp_on):
                el, nl = model(g)
                loss = criterion(el.float(), g["y"])
                if aw > 0:
                    nm = g["node_mask"]
                    ny = g["node_y"][nm]
                    npos = ny.sum()
                    pwn = torch.clamp((nm.sum().float() - npos) / torch.clamp(npos, min=1.0), max=cap)
                    loss = loss + aw * F.binary_cross_entropy_with_logits(
                        nl[nm].float(), ny, pos_weight=pwn)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            ep_loss += loss.item(); del g
        ep_loss /= len(TRAIN_W)

        val_ap = average_precision_score(vy, predict(VAL_W))
        scheduler.step(val_ap)
        hist["train_loss"].append(ep_loss); hist["val_ap"].append(float(val_ap))
        mark = ""
        if val_ap > best_ap:
            best_ap, left = val_ap, patience
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            mark = "  <- best"
        else:
            left -= 1
        if verbose:
            print(f"  seed{seed} ep {epoch:>2}/{epochs}  loss={ep_loss:.4f}  val PR-AUC={val_ap:.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.1e}  ({time.time()-t0:.0f}s){mark}", flush=True)
        if left == 0:
            if verbose:
                print("  조기종료 (검증 PR-AUC 개선 없음)")
            break

    model.load_state_dict(best_state)
    return dict(seed=seed, model=model, state=best_state, val_ap=float(best_ap), history=hist,
                val_prob=predict(VAL_W), test_prob=predict(TEST_W), pos_cap=cap, aux_w=aw)


# ── 평가 헬퍼 (§10-2 스윕과 §12 이후에서 공용) ──────────────────────────────
val_y, test_y = Y[n_tr:n_va], Y[n_va:]

def tuned_threshold(vy, vp):
    p_c, r_c, t_c = precision_recall_curve(vy, vp)
    f1_c = 2 * p_c * r_c / np.maximum(p_c + r_c, 1e-12)
    return float(t_c[int(np.nanargmax(f1_c[:-1]))])

def metrics(y_true, y_prob, thr):
    yp = (y_prob >= thr).astype(int)
    return dict(P=precision_score(y_true, yp, zero_division=0) * 100,
                R=recall_score(y_true, yp, zero_division=0) * 100,
                F1=f1_score(y_true, yp, zero_division=0) * 100)

def evaluate(vp, tp, name, vy=None, ty=None):
    '''[v7] vy/ty 를 넘기면 타 세트(전이 평가)에도 같은 프로토콜을 적용할 수 있다.'''
    vy = val_y if vy is None else vy
    ty = test_y if ty is None else ty
    thr_f1 = tuned_threshold(vy, vp)
    thr_bud = float(np.quantile(tp, 1 - ALERT_BUDGET))
    row = {"모델": name, "val PR-AUC": average_precision_score(vy, vp),
           "test PR-AUC": average_precision_score(ty, tp)}
    for tag, th in [("@0.5", 0.5), ("@F1튜닝", thr_f1), ("@예산0.1%", thr_bud)]:
        m = metrics(ty, tp, th)
        row |= {f"P{tag}": m["P"], f"R{tag}": m["R"], f"F1{tag}": m["F1"]}
    row["thr_F1"], row["thr_예산"] = thr_f1, thr_bud
    return row

### 10-2. [v7 신규 · 과제 ②] POS_WEIGHT_CAP 스윕 (5 / 15 / 30)

v5부터 미해결이던 과제입니다. §13 ablation과 같은 프로토콜(시드 42 고정, 동일 분할)로 cap만 바꿔 학습합니다.

> **선택 편향 방지:** 최적 cap은 **검증 PR-AUC로만** 고릅니다. 표의 test 컬럼은 참고용이며 선택에 쓰지 않습니다.
> 선택된 cap이 §11 본학습(4시드)과 §14 GBT에 그대로 적용됩니다.

In [ ]:
POS_SWEEP_ROWS, POS_SWEEP_BEST_RUN = [], None
if RUN_POS_SWEEP:
    print(f"POS_WEIGHT_CAP 스윕 {POS_CAP_SWEEP} — 시드 42 고정, 선택 기준 = val PR-AUC\n")
    _best_ap, _best_cap = -1.0, POS_WEIGHT_CAP
    for _cap in POS_CAP_SWEEP:
        _t0 = time.time()
        _r = train_one_seed(42, verbose=False, pos_cap=_cap)
        _row = evaluate(_r["val_prob"], _r["test_prob"], f"cap={_cap:g}")
        _row = {"cap": _cap, **{k: v for k, v in _row.items() if k != "모델"},
                "학습(분)": round((time.time() - _t0) / 60, 1)}
        POS_SWEEP_ROWS.append(_row)
        print(f"  cap={_cap:>4g}  val PR-AUC={_row['val PR-AUC']:.4f}  "
              f"(test PR-AUC={_row['test PR-AUC']:.4f}, F1@튜닝={_row['F1@F1튜닝']:.2f}%)  "
              f"{_row['학습(분)']}분", flush=True)
        if _r["val_ap"] > _best_ap:
            _best_ap, _best_cap = _r["val_ap"], _cap
            POS_SWEEP_BEST_RUN = {"val_prob": _r["val_prob"], "test_prob": _r["test_prob"],
                                  "pos_cap": _cap}
        del _r
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    _sw = pd.DataFrame(POS_SWEEP_ROWS)
    print("\n── 스윕 결과 ──")
    print(_sw[["cap", "val PR-AUC", "test PR-AUC", "F1@0.5", "F1@F1튜닝", "F1@예산0.1%", "학습(분)"]]
          .round(4).to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, colname, ttl in [(axes[0], "val PR-AUC", "검증 PR-AUC (선택 기준)"),
                             (axes[1], "test PR-AUC", "테스트 PR-AUC (참고용)")]:
        colors = ["#C44E52" if c == _best_cap else "#4C72B0" for c in _sw["cap"]]
        bars = ax.bar([f"cap={c:g}" for c in _sw["cap"]], _sw[colname], color=colors, width=0.55)
        for b, v in zip(bars, _sw[colname]):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom", fontsize=9)
        ax.set_title(ttl); ax.set_ylim(0, float(_sw[colname].max()) * 1.15)
        ax.grid(axis="y", lw=0.6, alpha=0.5); ax.set_axisbelow(True)
    plt.suptitle("POS_WEIGHT_CAP 스윕 (빨강 = 검증 기준 선택)", y=1.02)
    plt.tight_layout(); plt.show()

    POS_WEIGHT_CAP = float(_best_cap)
    print(f"\n>>> 선택: POS_WEIGHT_CAP = {POS_WEIGHT_CAP:g} (val PR-AUC {_best_ap:.4f}) — "
          f"§11 본학습과 §14 GBT에 적용됩니다.")
else:
    print(f"RUN_POS_SWEEP = False — 스윕 생략, POS_WEIGHT_CAP = {POS_WEIGHT_CAP:g} 유지.")

## 11. 다중 시드 학습 (SEED 0~3)

단일 시드 결과는 운이 섞입니다. 평균±표준편차 보고 + 시드 앙상블(분산을 줄이는 가장 저렴한 일반화 개선책).
[v7] §10-2에서 선택된 `POS_WEIGHT_CAP`과 계좌 보조 손실(`USE_NODE_AUX`)이 적용됩니다.

In [ ]:
t_all = time.time()
runs = []
for s in SEED_LIST:
    print(f"===== seed {s} (cap={POS_WEIGHT_CAP:g}, aux_w={AUX_NODE_W if USE_NODE_AUX else 0}) =====",
          flush=True)
    r = train_one_seed(s)
    print(f"  best val PR-AUC = {r['val_ap']:.4f}", flush=True)
    runs.append(r)
print(f"\n총 학습 시간 {(time.time()-t_all)/60:.1f}분")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for r in runs:
    axes[0].plot(r["history"]["train_loss"], "o-", ms=3, label=f"seed {r['seed']}")
    axes[1].plot(r["history"]["val_ap"], "s-", ms=3, label=f"seed {r['seed']}")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("train loss"); axes[0].set_title("학습 손실")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val PR-AUC"); axes[1].set_title("검증 PR-AUC")
for a in axes: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 12. 평가 — 소수클래스 Precision / Recall / F1 (테스트셋)

세 가지 임계값을 병행 보고합니다.

| 임계값 | 정의 | 왜 보는가 |
|---|---|---|
| **0.5** | 고정 | 팀원 GBT 프로토콜과 동일 조건 비교용 |
| **F1 튜닝** | 검증셋 F1 최대점 | v5와 동일. 단, 검증→테스트 분포 이동에 민감 |
| **경보 예산** | 테스트 거래의 상위 `ALERT_BUDGET`(0.1%)만 알림 | **운영 현실**: 조사 인력이 하루에 볼 수 있는 건수는 고정 |

In [ ]:
rows = [evaluate(r["val_prob"], r["test_prob"], f"GNN seed {r['seed']}") for r in runs]
ens_v = np.mean([r["val_prob"] for r in runs], axis=0)
ens_t = np.mean([r["test_prob"] for r in runs], axis=0)
row_ens = evaluate(ens_v, ens_t, f"GNN 앙상블({len(runs)}시드)")

res = pd.DataFrame(rows)
summary = pd.DataFrame([
    {"모델": f"GNN 단일(평균±표준편차, n={len(runs)})",
     **{c: f"{res[c].mean():.2f} ± {res[c].std(ddof=1):.2f}" for c in res.columns if c != "모델"}},
    {"모델": row_ens["모델"], **{c: f"{row_ens[c]:.2f}" for c in row_ens if c != "모델"}},
])
pd.set_option("display.width", 200, "display.max_columns", 50)
print("── 시드별 원시 결과 ──")
print(res.round(4).to_string(index=False))
print("\n── 요약 ──")
print(summary.to_string(index=False))
GNN_ENS, GNN_MEAN = row_ens, res

In [ ]:
# v5/v6 대비 개선폭 (v5 실행 기록: test PR-AUC 0.4846 / @0.5 F1 40.73 / @튜닝 F1 54.23)
V5 = {"test PR-AUC": 0.4846, "F1@0.5": 40.73, "F1@F1튜닝": 54.23}
cmp_rows = []
for k, v5v in V5.items():
    single = res[k].mean()
    cmp_rows.append({"지표": k, "v5_tuned_2 (단일시드)": v5v,
                     "v7 단일시드 평균": round(single, 4),
                     "v7 앙상블": round(row_ens[k], 4),
                     "앙상블 개선폭": round(row_ens[k] - v5v, 4)})
print(pd.DataFrame(cmp_rows).to_string(index=False))
print("\n(v6 앙상블 수치와의 비교는 v6 실행 기록을 아래 V6 dict에 채우면 함께 출력됩니다)")
V6 = {}   # 예: {"test PR-AUC": 0.51, "F1@F1튜닝": 56.1} — v6 노트북 §12 실행 기록에서 옮겨 적기
if V6:
    for k, v6v in V6.items():
        print(f"  {k}: v6 {v6v} → v7 {row_ens[k]:.4f} ({row_ens[k]-v6v:+.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
test_pred = (ens_t >= row_ens["thr_F1"]).astype(int)

tp_c, tr_c, _ = precision_recall_curve(test_y, ens_t)
axes[0].plot(tr_c, tp_c, color="#4C72B0", label=f"v7 앙상블 (AP={row_ens['test PR-AUC']:.4f})")
for r in runs:
    p_, r_, _ = precision_recall_curve(test_y, r["test_prob"])
    axes[0].plot(r_, p_, alpha=0.3, lw=1)
axes[0].axhline(0, color="k", lw=0)
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision"); axes[0].legend(fontsize=8)
axes[0].set_title("테스트 PR 곡선 (얇은 선 = 개별 시드)")

cm = confusion_matrix(test_y, test_pred)
axes[1].imshow(np.log1p(cm), cmap="Blues")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=11)
axes[1].set_xticks([0, 1], ["정상 예측", "세탁 예측"])
axes[1].set_yticks([0, 1], ["실제 정상", "실제 세탁"])
axes[1].set_title("혼동행렬 (앙상블, F1튜닝 임계값)")

axes[2].hist(ens_t[test_y == 0], bins=60, alpha=0.6, label="정상", color="#4C72B0", log=True)
axes[2].hist(ens_t[test_y == 1], bins=60, alpha=0.7, label="세탁", color="#C44E52", log=True)
axes[2].axvline(row_ens["thr_F1"], color="k", ls="--", label=f"F1튜닝 {row_ens['thr_F1']:.3f}")
axes[2].axvline(row_ens["thr_예산"], color="g", ls=":", label=f"예산 {row_ens['thr_예산']:.3f}")
axes[2].legend(fontsize=8); axes[2].set_title("예측 위험도 분포"); axes[2].set_xlabel("위험도(확률)")
plt.tight_layout(); plt.show()

print(f"기준선(전부 정상 예측) 정확도 = {(1-test_y.mean())*100:.4f}%")
print("주의: 세탁 비율이 극히 낮아 정확도는 모델 비교에 쓰지 마세요 (Precision/Recall/F1/PR-AUC 사용).")

### 12-2. [v7 신규 · 과제 ③] 계좌 단위 보조 판정 (화두 1)

개별 거래 라벨의 노이즈는 거래 단위 모델로는 못 없앱니다. 그래서 **판정의 최종 단위를 계좌로 끌어올린**
보조 지표를 병행합니다 — 실무(조사·STR)도 결국 계좌/고객 단위로 진행되므로 운영 관점과도 일치합니다.

* **계좌 점수** = 그 계좌가 관여(송·수신)한 테스트 거래 위험도의 **최댓값**과 **상위 3건 평균** 두 가지
* **계좌 라벨** = 테스트 구간에서 세탁 거래에 1건이라도 관여했는가
* 거래 단위와 동일한 **경보 예산 0.1%** 를 계좌 수 기준으로 적용해 비교

상위 3건 평균이 최댓값보다 PR-AUC가 높다면 "한 건의 튀는 점수"보다 "반복된 의심 행위"가 신호라는 뜻이며,
이는 라벨 노이즈 대응이 실제로 작동함을 의미합니다.

In [ ]:
tdf = df.iloc[n_va:]
acct2 = np.concatenate([tdf["src_id"].to_numpy(), tdf["dst_id"].to_numpy()])
prob2 = np.concatenate([ens_t, ens_t])
lab2  = np.concatenate([test_y, test_y])

A = pd.DataFrame({"acct": acct2, "p": prob2, "y": lab2})
acct_df = A.groupby("acct").agg(score_max=("p", "max"), n_tx=("p", "size"), y=("y", "max"))
top3 = (A.sort_values("p", ascending=False).groupby("acct").head(3)
        .groupby("acct")["p"].mean().rename("score_top3"))
acct_df = acct_df.join(top3)
n_acct = len(acct_df)
acct_y = acct_df["y"].to_numpy()
print(f"테스트 구간 등장 계좌 {n_acct:,}개, 세탁 관여 계좌 {int(acct_y.sum()):,}개 "
      f"({acct_y.mean()*100:.4f}%)")

acct_rows = []
for colname, tag in [("score_max", "계좌(최댓값)"), ("score_top3", "계좌(상위3 평균)")]:
    sc = acct_df[colname].to_numpy()
    ap_ = average_precision_score(acct_y, sc)
    thr_b = float(np.quantile(sc, 1 - ALERT_BUDGET))
    yp = (sc >= thr_b).astype(int)
    acct_rows.append({"판정 단위": tag, "PR-AUC": ap_,
                      "P@예산0.1%": precision_score(acct_y, yp, zero_division=0) * 100,
                      "R@예산0.1%": recall_score(acct_y, yp, zero_division=0) * 100,
                      "경보 수": int(yp.sum())})
# 거래 단위 비교 행
acct_rows.append({"판정 단위": "거래(§12 앙상블)", "PR-AUC": row_ens["test PR-AUC"],
                  "P@예산0.1%": row_ens["P@예산0.1%"], "R@예산0.1%": row_ens["R@예산0.1%"],
                  "경보 수": int((ens_t >= row_ens["thr_예산"]).sum())})
acct_res = pd.DataFrame(acct_rows)
print("\n── 판정 단위 비교 (동일 예산 0.1%) ──")
print(acct_res.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for colname, lab_, c_ in [("score_max", "계좌(최댓값)", "#C44E52"),
                          ("score_top3", "계좌(상위3 평균)", "#DD8452")]:
    p_, r_, _ = precision_recall_curve(acct_y, acct_df[colname].to_numpy())
    axes[0].plot(r_, p_, color=c_, label=f"{lab_} (AP={average_precision_score(acct_y, acct_df[colname]):.4f})")
p_, r_, _ = precision_recall_curve(test_y, ens_t)
axes[0].plot(r_, p_, color="#4C72B0", alpha=0.7, label=f"거래 단위 (AP={row_ens['test PR-AUC']:.4f})")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("판정 단위별 PR 곡선 — 유병률이 달라 절대값 직접 비교는 참고용")
axes[0].legend(fontsize=8)

nb_ = axes[1]
grp = acct_df.groupby(pd.cut(acct_df["n_tx"], [0, 1, 3, 10, 30, np.inf]),
                      observed=True)["y"].mean() * 100
nb_.bar([str(i) for i in grp.index], grp.values, color="#8172B2")
nb_.set_xlabel("계좌의 테스트 구간 거래 수"); nb_.set_ylabel("세탁 관여율(%)")
nb_.set_title("거래 수별 세탁 관여율 — 집계 판정의 근거")
plt.tight_layout(); plt.show()

ACCT_RESULT = acct_res.to_dict("records")
print("\n해석 가이드: 상위3 평균의 PR-AUC가 최댓값보다 높으면 '반복 의심 행위' 신호가 유효 →")
print("운영에서는 거래 경보와 계좌 경보를 병행하고, 계좌 경보를 조사 케이스 생성 단위로 쓰는 것을 권장.")

## 13. 어떤 개선이 얼마나 기여했나 (ablation)

각 개선안을 **하나씩 누적**하며 동일 시드(42)·동일 프로토콜로 학습해 비교합니다.

> **수치는 하드코딩하지 않습니다.** `RUN_ABLATION = True` 로 두고 실행했을 때 직접 측정한 값만 표시합니다.

- [v7] 사다리 끝에 **`+ 계좌 보조 손실`** 단계를 추가 — 멀티태스크의 기여분을 분리 측정합니다.
- POS_WEIGHT_CAP 스윕은 §10-2로 분리했습니다 (여기서는 §10-2가 선택한 cap을 공통 사용).
- **상대 시간차 OFF** 는 §7 재실행 없이 해당 피처 컬럼을 0으로 마스킹해 측정합니다(입력 차원 유지).
- 설정을 바꿀 때마다 윈도우 캐시를 비웁니다. 실행 후 `finally` 블록에서 반드시 v7 전체 설정으로 복구됩니다.

In [ ]:
# ── §13 ablation 설정 ────────────────────────────────────────────────────────
RUN_ABLATION  = False    # True 로 바꾸고 이 셀 + 아래 셀을 실행하면 실제 측정
ABLATION_SEED = 42       # 단일 시드로 비교 — 시드 효과를 제거해 "설정 차이"만 봅니다
ABLATION      = []       # 측정 결과가 여기에 누적됩니다

# 개선안을 하나씩 누적하며 켜는 순서
# (t=상대시간차, p=포트번호, m=MEGA집계, nf=노드피처수, w=윈도우, c=컨텍스트, a=[v7]계좌보조손실)
ABL_CONFIGS = [
    ("v5 기준선",         dict(t=False, p=False, m=False, nf=4, w=200_000, c=0,       a=False)),
    ("+ 상대 시간차",      dict(t=True,  p=False, m=False, nf=4, w=200_000, c=0,       a=False)),
    ("+ 노드피처 9종",     dict(t=True,  p=False, m=False, nf=9, w=200_000, c=0,       a=False)),
    ("+ 포트 번호",        dict(t=True,  p=True,  m=False, nf=9, w=200_000, c=0,       a=False)),
    ("+ MEGA 집계",       dict(t=True,  p=True,  m=True,  nf=9, w=200_000, c=0,       a=False)),
    ("+ 컨텍스트 윈도우",   dict(t=True,  p=True,  m=True,  nf=9, w=100_000, c=100_000, a=False)),  # = v6 전체
    ("+ 계좌 보조 손실",    dict(t=True,  p=True,  m=True,  nf=9, w=100_000, c=100_000, a=True)),   # = v7 전체
]

ABL_SHOW_COLS = ["설정", "val PR-AUC", "test PR-AUC", "F1@0.5", "F1@F1튜닝", "F1@예산0.1%", "학습(분)"]


def show_ablation(rows, cols=(("test PR-AUC", "테스트 PR-AUC"),
                              ("F1@F1튜닝", "테스트 F1 (튜닝 임계값)"))):
    '''측정 결과가 있으면 표+막대그래프를, 없으면 안내만 출력한다 (빈 리스트에도 죽지 않음).'''
    if not rows:
        print("ablation 미실행 — 위의 RUN_ABLATION 을 True 로 바꾸고 이 셀과 아래 셀을 실행하십시오.")
        print(f"  설정 {len(ABL_CONFIGS)}개를 시드 {ABLATION_SEED} 로 각각 학습합니다 (설정당 수 분).")
        return None

    abl = pd.DataFrame(rows)
    print(abl[[c for c in ABL_SHOW_COLS if c in abl.columns]].round(4).to_string(index=False))

    missing = [c for c, _ in cols if c not in abl.columns]
    if missing:
        print(f"\n(그래프 생략 — 컬럼 없음: {missing})")
        return abl

    fig, axes = plt.subplots(1, len(cols), figsize=(7 * len(cols), 4.4))
    axes = np.atleast_1d(axes)
    xs = np.arange(len(abl))
    for ax, (col, ttl) in zip(axes, cols):
        base = abl[col].iloc[0]
        colors = ["#898781"] + ["#2a78d6" if v >= base else "#e34948" for v in abl[col].iloc[1:]]
        bars = ax.bar(xs, abl[col], color=colors, width=0.6)
        ax.axhline(base, color="#0b0b0b", ls="--", lw=1, label=f"기준선 {base:.4f}")
        for b, v in zip(bars, abl[col]):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9)
        ax.set_xticks(xs); ax.set_xticklabels(abl["설정"], rotation=25, ha="right", fontsize=8)
        ax.set_ylim(0, float(abl[col].max()) * 1.18)
        ax.grid(axis="y", lw=0.6, color="#e1e0d9"); ax.set_axisbelow(True)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
        ax.set_title(ttl); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    return abl


show_ablation(ABLATION)

In [ ]:
if RUN_ABLATION:
    # 상대 시간차 OFF 는 §7 재실행 없이 해당 컬럼을 0으로 마스킹하면 동치입니다(입력 차원 유지).
    _EF_FULL, _EF_NOTIME = MAIN["EF"], None
    _TIME_IDX = [EDGE_FEATURES.index(c) for c in TIME_FEATURES]

    def _set_time_feats(on):
        global EF, _EF_NOTIME
        if on:
            MAIN["EF"] = EF = _EF_FULL
        else:
            if _EF_NOTIME is None:
                _EF_NOTIME = _EF_FULL.copy()
                if _TIME_IDX:
                    _EF_NOTIME[:, _TIME_IDX] = 0.0
            MAIN["EF"] = EF = _EF_NOTIME

    _SAVED = dict(t=USE_TIME_FEATS, p=USE_PORTS, m=USE_MEGA,
                  nf=NODE_FEAT_DIM, w=WINDOW_SIZE, c=CONTEXT_EDGES)
    ABLATION.clear()
    print(f"ablation 시작 — 설정 {len(ABL_CONFIGS)}개 × 시드 {ABLATION_SEED} (cap={POS_WEIGHT_CAP:g})\n")
    try:
        for _name, _cfg in ABL_CONFIGS:
            _t0 = time.time()
            _set_time_feats(_cfg["t"])
            globals().update(USE_TIME_FEATS=_cfg["t"], USE_PORTS=_cfg["p"], USE_MEGA=_cfg["m"],
                             NODE_FEAT_DIM=_cfg["nf"], WINDOW_SIZE=_cfg["w"], CONTEXT_EDGES=_cfg["c"])
            globals().update(EDGE_EXTRA=2 + (2 if _cfg["p"] else 0))
            globals().update(EDGE_DIM=len(EDGE_FEATURES) + EDGE_EXTRA)
            globals().update(TRAIN_W=window_ranges(0, n_tr),
                             VAL_W=window_ranges(n_tr, n_va),
                             TEST_W=window_ranges(n_va, n))
            MAIN["cache"].clear()    # 창 정의·엣지 피처가 바뀌었으므로 캐시 무효화 (필수)

            _r = train_one_seed(ABLATION_SEED, verbose=False,
                                aux_w=(AUX_NODE_W if _cfg.get("a") else 0.0))
            _row = evaluate(_r["val_prob"], _r["test_prob"], _name)
            _row.pop("모델", None)
            _row = {"설정": _name, **_row, "학습(분)": round((time.time() - _t0) / 60, 1)}
            ABLATION.append(_row)
            print(f"[{len(ABLATION)}/{len(ABL_CONFIGS)}] {_name:<16s} "
                  f"test PR-AUC={_row['test PR-AUC']:.4f}  "
                  f"F1@튜닝={_row['F1@F1튜닝']:.2f}%  ({_row['학습(분)']}분)", flush=True)
            del _r
    finally:
        # 예외가 나도 반드시 v7 전체 설정으로 되돌립니다 (이후 셀이 오염되지 않도록)
        MAIN["EF"] = EF = _EF_FULL
        _EF_NOTIME = None
        globals().update(USE_TIME_FEATS=_SAVED["t"], USE_PORTS=_SAVED["p"], USE_MEGA=_SAVED["m"],
                         NODE_FEAT_DIM=_SAVED["nf"], WINDOW_SIZE=_SAVED["w"], CONTEXT_EDGES=_SAVED["c"])
        EDGE_EXTRA = 2 + (2 if USE_PORTS else 0)
        EDGE_DIM   = len(EDGE_FEATURES) + EDGE_EXTRA
        TRAIN_W = window_ranges(0, n_tr)
        VAL_W   = window_ranges(n_tr, n_va)
        TEST_W  = window_ranges(n_va, n)
        MAIN["cache"].clear()
        print("\n설정 복구 완료 — 이후 셀은 v7 전체 설정으로 실행됩니다.")

    abl_df = show_ablation(ABLATION)

    # 누적 기여도 (직전 설정 대비 증분)
    if len(ABLATION) > 1:
        _d = pd.DataFrame(ABLATION)
        _d["Δ test PR-AUC"] = _d["test PR-AUC"].diff().round(4)
        _d["Δ F1@F1튜닝"]   = _d["F1@F1튜닝"].diff().round(2)
        print("\n── 직전 설정 대비 증분 기여도 ──")
        print(_d[["설정", "test PR-AUC", "Δ test PR-AUC", "F1@F1튜닝", "Δ F1@F1튜닝"]]
              .to_string(index=False))
else:
    print("RUN_ABLATION = False — ablation 미실행.")
    print("측정하려면 위 셀에서 RUN_ABLATION = True 로 바꾼 뒤 두 셀을 순서대로 실행하십시오.")

### 13-2. [v7 신규 · 과제 ⑤] 상대 시간차 피처 제거 A/B — 실데이터 이관 대비

상대 시간차 피처(Δt·최근 빈도)는 **합성 데이터 생성기의 시간 패턴에 맞춰졌을 가능성**이 있습니다.
실데이터로 이관하기 전에 "이 피처군에 성능이 얼마나 의존하는가"를 반드시 수치로 확보해야 합니다.

* **A (포함)**: 현재 전체 설정 — §10-2 스윕이 돌았으면 그 최적-cap 시드42 결과를 재사용(추가 학습 없음)
* **B (제거)**: 시간차 피처 컬럼을 0으로 마스킹하고 **재학습** (평가 시만 마스킹하는 것은 학습 분포와 어긋나
  의존도를 과대 추정하므로, 반드시 재학습으로 비교합니다)

**판정 기준** — test PR-AUC 상대 하락폭 기준:
| 하락폭 | 해석 | 이관 시 조치 |
|---|---|---|
| < 5% | 의존도 낮음 | 피처 유지 가능, 이관 후 모니터링만 |
| 5~15% | 중간 의존 | 이관 초기 두 버전 병행 운영(챔피언-챌린저) 후 실데이터로 재판정 |
| > 15% | 강한 의존 | **합성 패턴 과적합 의심** — 실데이터 학습 전 피처 재설계 또는 제거를 기본안으로 |

In [ ]:
TIME_AB_RESULT = None
if RUN_TIME_AB and USE_TIME_FEATS and TIME_FEATURES:
    _TIME_IDX = [EDGE_FEATURES.index(c) for c in TIME_FEATURES]

    # A: 시간차 포함 — 스윕 최적 실행이 있으면 재사용, 없으면 신규 학습
    if POS_SWEEP_BEST_RUN is not None:
        a_val, a_test = POS_SWEEP_BEST_RUN["val_prob"], POS_SWEEP_BEST_RUN["test_prob"]
        a_src = f"§10-2 스윕 최적(cap={POS_WEIGHT_CAP:g}) 시드42 재사용"
    else:
        print("A(포함) 학습 중...", flush=True)
        _rA = train_one_seed(42, verbose=False)
        a_val, a_test = _rA["val_prob"], _rA["test_prob"]; a_src = "신규 학습 (시드42)"
        del _rA

    # B: 시간차 제거 후 재학습
    _EF_FULL = MAIN["EF"]
    try:
        print(f"B(제거) 학습 중... (A: {a_src})", flush=True)
        _ef_masked = _EF_FULL.copy()
        _ef_masked[:, _TIME_IDX] = 0.0
        MAIN["EF"] = EF = _ef_masked
        MAIN["cache"].clear()
        _rB = train_one_seed(42, verbose=False)
        b_val, b_test = _rB["val_prob"], _rB["test_prob"]
        del _rB
    finally:
        MAIN["EF"] = EF = _EF_FULL
        MAIN["cache"].clear()
    del _ef_masked

    rowA = evaluate(a_val, a_test, "A: 시간차 포함")
    rowB = evaluate(b_val, b_test, "B: 시간차 제거(재학습)")
    ab = pd.DataFrame([rowA, rowB])
    print("\n── 상대 시간차 A/B (시드 42, 동일 프로토콜) ──")
    print(ab[["모델", "val PR-AUC", "test PR-AUC", "F1@0.5", "F1@F1튜닝", "F1@예산0.1%"]]
          .round(4).to_string(index=False))

    _apA, _apB = rowA["test PR-AUC"], rowB["test PR-AUC"]
    rel = (_apA - _apB) / max(_apA, 1e-9)
    if rel < 0.05:
        verdict = "의존도 낮음 (<5%) — 피처 유지 가능, 이관 후 모니터링만"
    elif rel < 0.15:
        verdict = "중간 의존 (5~15%) — 이관 초기 챔피언-챌린저 병행 운영 권장"
    else:
        verdict = "강한 의존 (>15%) — 합성 패턴 과적합 의심, 실데이터 학습 전 재설계/제거 기본안"
    print(f"\ntest PR-AUC: 포함 {_apA:.4f} → 제거 {_apB:.4f} "
          f"(하락 {_apA-_apB:+.4f}, 상대 {rel*100:.1f}%)")
    print(f">>> 판정: {verdict}")

    TIME_AB_RESULT = {"a_source": a_src,
                      "pr_auc_with_time": float(_apA), "pr_auc_without_time": float(_apB),
                      "abs_drop": float(_apA - _apB), "rel_drop": float(rel),
                      "f1_tuned_with": float(rowA["F1@F1튜닝"]), "f1_tuned_without": float(rowB["F1@F1튜닝"]),
                      "verdict": verdict}
else:
    print("RUN_TIME_AB = False (또는 시간차 피처 미사용) — A/B 생략.")

## 14. 팀원 GBT 방식과 교차 검증 (동일 분할·동일 지표)

동일 분할·동일 피처로 GBT를 직접 학습해 "GNN이 좋아서 이긴 것"과 "피처가 좋아서 이긴 것"을 분리합니다.

* GBT는 **거래 1건의 피처만** 봅니다 (그래프 구조 없음). GNN과의 차이 = **그래프 구조의 순수 기여분**.
* [v7] `class_weight` 에 §10-2가 선택한 `POS_WEIGHT_CAP` 이 반영됩니다.

> 팀원의 실제 GFP 결과 CSV가 있으면 `TEAM_GBT_RESULT` 에 넣어 같은 표에 올리세요.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

t0 = time.time()
X = EF                                   # GNN과 완전히 동일한 엣지 피처
y_int = Y.astype(int)
gbt = HistGradientBoostingClassifier(max_iter=500, learning_rate=0.1, max_leaf_nodes=63,
                                     l2_regularization=1.0, class_weight={0: 1.0, 1: POS_WEIGHT_CAP},
                                     early_stopping=False, random_state=42)
gbt.fit(X[:n_tr], y_int[:n_tr])
gbt_v = gbt.predict_proba(X[n_tr:n_va])[:, 1]
gbt_t = gbt.predict_proba(X[n_va:])[:, 1]
print(f"GBT 학습 {time.time()-t0:.0f}s")

row_gbt = evaluate(gbt_v, gbt_t, "GBT (그래프 없음)")

TEAM_GBT_RESULT = None    # 예: {"모델": "팀원 GBT+GFP", "F1@0.5": 47.0, ...}
tbl = [row_gbt,
       {**{k: (res[k].mean() if k != "모델" else "GNN 단일 평균") for k in row_gbt}},
       row_ens]
if TEAM_GBT_RESULT:
    tbl.append(TEAM_GBT_RESULT)
comp = pd.DataFrame(tbl)[["모델", "test PR-AUC", "F1@0.5", "P@F1튜닝", "R@F1튜닝", "F1@F1튜닝", "F1@예산0.1%"]]
print(comp.round(4).to_string(index=False))
print(f"\n그래프 구조의 순수 기여분(PR-AUC): {row_ens['test PR-AUC'] - row_gbt['test PR-AUC']:+.4f}")

## 15. [v7 개편 · 과제 ①] 세트 간 비교 + zero-shot 전이 평가 (화두 7)

v6은 스캐폴드(비교표 1행)였습니다. v7은 **실제 일반화 검증**을 수행합니다:

1. **세트 검출** — `IBM_AML_dataset/` 폴더 또는 zip에서 사용 가능한 세트를 자동 확인
2. **zero-shot 전이** — **HI-Small로 학습한 앙상블을 그대로** 타 세트에 적용해 채점.
   재학습 없이 얼마나 버티는지가 일반화의 가장 엄격한 척도입니다.
3. **공정 비교 장치**
   * 피처 파이프라인(§6·§7의 함수들)을 타깃 세트에 동일하게 적용하되, **범주 어휘(pf/pc/rc)는 학습 세트 것을
     사용**해 입력 차원을 일치시킵니다 (미등장 범주는 전부 0 → 모델이 "모르는 유형"으로 취급)
   * 통화 정규화 통계만 타깃 세트의 train 구간으로 적합합니다 — 모델 학습에는 쓰지 않으므로 누수가 아니며,
     운영에서도 신규 기관 데이터에 자체 정규화를 적용하는 것과 같은 조건입니다
   * 세트마다 세탁 유병률이 달라 PR-AUC 절대값 비교는 불공정 → **lift = PR-AUC ÷ 유병률** 를 함께 보고
   * F1 임계값은 타깃 세트의 val 구간으로 튜닝(모델은 불변 — 임계값 캘리브레이션은 운영상 허용되는 절차)

> **파일이 없으면 해당 세트는 자동 스킵**되고 안내만 출력됩니다. `LI-Small_Trans.csv` 를
> `IBM_AML_dataset/` 에 올린 뒤 이 셀만 다시 실행하면 됩니다. Medium 세트(수천만 건)는 메모리 확인 후
> `TRANSFER_SETS` 에 추가하세요. 완전한 세트별 재학습 비교는 `DATASET` 을 바꿔 노트북 전체를 재실행해
> 이 표에 추가하는 방식을 유지합니다(모델 재학습이 필요하므로 자동화하지 않음).

In [ ]:
CANDIDATE_SETS = ["HI-Small", "LI-Small", "HI-Medium", "LI-Medium"]
available, missing = [], []
for ds in CANDIDATE_SETS:
    p = Path("IBM_AML_dataset") / f"{ds}_Trans.csv"
    in_zip = zf is not None and any(nm.endswith(f"{ds}_Trans.csv") for nm in zf.namelist())
    (available if (p.exists() or in_zip) else missing).append(ds)
print("사용 가능:", available)
print("없음     :", missing, "→ 해당 세트는 스킵합니다.")

ens_models = [r["model"] for r in runs]


def prepare_bundle(ds):
    '''[v7] 타 세트를 본학습과 동일 파이프라인으로 정제·피처화해 번들로 반환 (zero-shot 채점용).'''
    raw = load_transactions(ds)
    d2, _meta = clean_frame(raw, verbose=False)
    m = len(d2)
    m_tr, m_va = int(m * TRAIN_FRAC), int(m * (TRAIN_FRAC + VAL_FRAC))
    d2["order_idx"] = np.arange(m, dtype=np.int64)
    is_tr2 = np.zeros(m, bool); is_tr2[:m_tr] = True

    sk = d2["From Bank"].astype(str) + "_" + d2["Account"].astype(str)
    dk = d2["To Bank"].astype(str) + "_" + d2["Account.1"].astype(str)
    codes2, _ = pd.factorize(pd.concat([sk, dk], ignore_index=True))
    d2["src_id"] = codes2[:m].astype(np.int64)
    d2["dst_id"] = codes2[m:].astype(np.int64)
    d2["self_loop"] = (d2["src_id"] == d2["dst_id"]).astype(np.float32)

    # 범주 어휘는 학습 세트 것을 강제(입력 차원 일치) — 정규화 통계만 타깃 train 구간으로 적합
    d2, base2, _ = add_base_features(d2, is_tr2, cats=(PF_CATS, PC_CATS, RC_CATS))
    d2, tf2 = add_time_features(d2)
    feats2 = base2 + tf2
    assert feats2 == EDGE_FEATURES, "피처 이름/순서가 학습 세트와 다릅니다 — 전이 평가 불가"

    return dict(SRC=d2["src_id"].to_numpy(), DST=d2["dst_id"].to_numpy(),
                Y=d2["Is Laundering"].to_numpy().astype(np.float32),
                EF=np.ascontiguousarray(d2[feats2].to_numpy(dtype=np.float32)),
                AP=d2["amt_paid_z"].to_numpy(), AR=d2["amt_recv_z"].to_numpy(),
                n=m, n_tr=m_tr, n_va=m_va, cache={})


@torch.no_grad()
def predict_bundle(models, wins, B):
    '''앙상블 평균 확률 — 번들 윈도우를 캐시 없이 1회 순회 (메모리 절약).'''
    amp_on = USE_AMP and DEVICE.type == "cuda"
    ps = []
    for w in wins:
        g = to_dev(build_window(*w, D=B))
        outs = []
        for mdl in models:
            mdl.eval()
            with torch.autocast("cuda", torch.float16, enabled=amp_on):
                el, _ = mdl(g)
            outs.append(torch.sigmoid(el.float()).cpu().numpy())
        ps.append(np.mean(outs, axis=0))
        del g
    return np.concatenate(ps)


# ── 비교표: 학습 세트(in-set) + 타 세트(zero-shot) ──
_prev_in = float(test_y.mean())
TRANSFER_ROWS = [{"세트": DATASET, "평가": "in-set (학습 세트)", "거래 수": int(n),
                  "세탁 유병률(%)": round(_prev_in * 100, 4),
                  "test PR-AUC": round(float(row_ens["test PR-AUC"]), 4),
                  "lift(=AP/유병률)": round(float(row_ens["test PR-AUC"]) / max(_prev_in, 1e-9), 1),
                  "F1@F1튜닝": round(float(row_ens["F1@F1튜닝"]), 2),
                  "F1@예산0.1%": round(float(row_ens["F1@예산0.1%"]), 2)}]

if RUN_TRANSFER:
    targets = [s for s in TRANSFER_SETS if s in available and s != DATASET]
    skipped = [s for s in TRANSFER_SETS if s not in available]
    if skipped:
        print(f"\n스킵(파일 없음): {skipped}")
    for ds in targets:
        print(f"\n===== {ds} zero-shot 전이 평가 =====", flush=True)
        try:
            _t0 = time.time()
            B = prepare_bundle(ds)
            vw = window_ranges(B["n_tr"], B["n_va"])
            tw = window_ranges(B["n_va"], B["n"])
            vp = predict_bundle(ens_models, vw, B)
            tp = predict_bundle(ens_models, tw, B)
            vy2 = B["Y"][B["n_tr"]:B["n_va"]]
            ty2 = B["Y"][B["n_va"]:]
            row = evaluate(vp, tp, ds, vy=vy2, ty=ty2)
            prev2 = float(ty2.mean())
            TRANSFER_ROWS.append({"세트": ds, "평가": "zero-shot (HI-Small 학습 앙상블)",
                                  "거래 수": int(B["n"]),
                                  "세탁 유병률(%)": round(prev2 * 100, 4),
                                  "test PR-AUC": round(float(row["test PR-AUC"]), 4),
                                  "lift(=AP/유병률)": round(float(row["test PR-AUC"]) / max(prev2, 1e-9), 1),
                                  "F1@F1튜닝": round(float(row["F1@F1튜닝"]), 2),
                                  "F1@예산0.1%": round(float(row["F1@예산0.1%"]), 2)})
            print(f"  test PR-AUC={row['test PR-AUC']:.4f} (유병률 {prev2*100:.4f}%), "
                  f"F1@튜닝={row['F1@F1튜닝']:.2f}%  ({(time.time()-_t0)/60:.1f}분)")
            del B, vp, tp
        except FileNotFoundError as e:
            print(f"  스킵: {e}")
        except MemoryError:
            print(f"  스킵: 메모리 부족 — {ds} 는 더 큰 메모리 환경에서 실행하세요.")

cross = pd.DataFrame(TRANSFER_ROWS)
print("\n── 세트 간 비교표 (화두 7) ──")
print(cross.to_string(index=False))

if len(cross) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    colors = ["#4C72B0" if "in-set" in e else "#DD8452" for e in cross["평가"]]
    for ax, colname, ttl in [(axes[0], "test PR-AUC", "테스트 PR-AUC (유병률 차이 주의)"),
                             (axes[1], "lift(=AP/유병률)", "lift = PR-AUC ÷ 유병률 (세트 간 공정 비교)")]:
        bars = ax.bar(cross["세트"], cross[colname], color=colors, width=0.5)
        for b, v in zip(bars, cross[colname]):
            ax.text(b.get_x() + b.get_width() / 2, v, f"{v}", ha="center", va="bottom", fontsize=9)
        ax.set_title(ttl); ax.grid(axis="y", lw=0.6, alpha=0.5); ax.set_axisbelow(True)
    plt.suptitle("파랑 = in-set, 주황 = zero-shot 전이", y=1.02)
    plt.tight_layout(); plt.show()

    _zs = cross[cross["평가"].str.startswith("zero")]
    if len(_zs):
        _keep = _zs["lift(=AP/유병률)"].max() / max(cross.iloc[0]["lift(=AP/유병률)"], 1e-9)
        print(f"\nzero-shot lift 유지율(최고 세트 기준): {_keep*100:.1f}% — "
              "높을수록 학습 세트 특이 패턴이 아닌 일반 신호를 학습했다는 뜻입니다.")
else:
    print("\n타 세트 파일이 없어 in-set 1행만 표시됩니다. LI-Small_Trans.csv 를 IBM_AML_dataset/ 에 "
          "올린 뒤 이 셀만 재실행하세요 (모델 재학습 불필요 — 저장된 앙상블로 즉시 채점).")

## 16. [v7 신규 · 과제 ④] 드리프트 모니터링 — 검증→테스트 하락폭의 운영 지표화

이 노트북의 시간 순 분할에서 **val→test 성능 하락폭은 "미래 분포 이동에 대한 민감도"의 대리 지표**입니다.
운영에서는 같은 계산을 "직전 평가 구간 → 최신 구간"으로 반복하면 그대로 드리프트 경보가 됩니다.

| 지표 | 임계값 | 운영 조치 |
|---|---|---|
| **PR-AUC 상대 하락** | 주의 ≥ 10%, 경보 ≥ 20% | 주의: 원인 분석 착수 / 경보: 재학습 트리거 + 임계값 재캘리브레이션 |
| **점수 분포 PSI** (val→test) | 주의 ≥ 0.10, 경보 ≥ 0.25 | 입력/점수 분포 이동 감지 — 성능 라벨이 늦게 도착해도 먼저 울리는 조기 경보 |
| **경보율 안정성** | 예산 0.1%의 0.5~2배 이탈 시 주의 | 검증 임계값 고정 시 실제 경보량 폭주/고갈 감지 → 예산 기반 임계값 사용 근거 |
| **윈도우별 롤링 PR-AUC** | 추세 감시 | 하락이 특정 구간 국소적인지 전역적인지 구분 |

라벨이 늦게 확정되는 AML 특성상 **PSI·경보율(라벨 불필요)** 를 1차 경보로, **PR-AUC(라벨 필요)** 를
2차 확정 지표로 쓰는 2단 구성을 권장합니다.

In [ ]:
def psi(expected, actual, bins=10):
    '''Population Stability Index — expected(기준) 분위 구간에 대한 actual 분포 이동량.'''
    qs = np.quantile(expected, np.linspace(0, 1, bins + 1))
    qs[0], qs[-1] = -np.inf, np.inf
    e = np.histogram(expected, qs)[0] / max(len(expected), 1)
    a = np.histogram(actual, qs)[0] / max(len(actual), 1)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))


def drift_status(rel_drop):
    if rel_drop >= DRIFT_ALERT: return "🔴 경보"
    if rel_drop >= DRIFT_WARN:  return "🟡 주의"
    return "🟢 정상"


# ── (1) 시드별 + 앙상블 val→test PR-AUC 하락 ──
drift_rows = []
for r in runs:
    va_ = average_precision_score(val_y, r["val_prob"])
    te_ = average_precision_score(test_y, r["test_prob"])
    rel = (va_ - te_) / max(va_, 1e-9)
    drift_rows.append({"모델": f"seed {r['seed']}", "val PR-AUC": va_, "test PR-AUC": te_,
                       "하락": va_ - te_, "상대 하락(%)": rel * 100, "상태": drift_status(rel)})
_va = average_precision_score(val_y, ens_v)
_te = average_precision_score(test_y, ens_t)
_rel_ens = (_va - _te) / max(_va, 1e-9)
drift_rows.append({"모델": "앙상블", "val PR-AUC": _va, "test PR-AUC": _te,
                   "하락": _va - _te, "상대 하락(%)": _rel_ens * 100, "상태": drift_status(_rel_ens)})
drift_df = pd.DataFrame(drift_rows)
print("── val→test PR-AUC 하락 (주의 ≥ {:.0f}%, 경보 ≥ {:.0f}%) ──".format(DRIFT_WARN*100, DRIFT_ALERT*100))
print(drift_df.round(4).to_string(index=False))

# ── (2) 점수 분포 PSI (라벨 불필요 — 조기 경보용) ──
score_psi = psi(ens_v, ens_t)
psi_status = "🔴 경보" if score_psi >= PSI_ALERT else ("🟡 주의" if score_psi >= PSI_WARN else "🟢 정상")
print(f"\n점수 분포 PSI (val→test) = {score_psi:.4f}  [{psi_status}] "
      f"(주의 ≥ {PSI_WARN}, 경보 ≥ {PSI_ALERT})")

# ── (3) 경보율 안정성 — 검증 임계값 고정 시 테스트에서 경보가 예산대로 나오는가 ──
thr_val_budget = float(np.quantile(ens_v, 1 - ALERT_BUDGET))
realized = float((ens_t >= thr_val_budget).mean())
ratio = realized / ALERT_BUDGET
rate_status = "🟢 정상" if 0.5 <= ratio <= 2.0 else "🟡 주의"
print(f"경보율 안정성: 검증 임계값 {thr_val_budget:.4f} 고정 시 테스트 경보율 "
      f"{realized*100:.4f}% (예산 {ALERT_BUDGET*100:.1f}%의 {ratio:.2f}배) [{rate_status}]")
print("  → 비율이 1에서 멀수록 점수 분포가 이동 중 — 예산(분위수) 기반 임계값 운영 권장 근거.")

# ── (4) 테스트 윈도우별 롤링 PR-AUC ──
roll = []
for (c_, lo_, hi_) in TEST_W:
    yw = Y[lo_:hi_]
    pw_ = ens_t[lo_ - n_va:hi_ - n_va]
    if yw.sum() >= 5:                          # 양성 5건 미만 구간은 PR-AUC가 무의미
        roll.append({"윈도우 시작": lo_, "양성 수": int(yw.sum()),
                     "PR-AUC": average_precision_score(yw, pw_)})
roll_df = pd.DataFrame(roll)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
xs = np.arange(len(drift_df))
axes[0].bar(xs - 0.18, drift_df["val PR-AUC"], width=0.36, color="#4C72B0", label="val")
axes[0].bar(xs + 0.18, drift_df["test PR-AUC"], width=0.36, color="#DD8452", label="test")
axes[0].set_xticks(xs); axes[0].set_xticklabels(drift_df["모델"], fontsize=9)
axes[0].set_title("val vs test PR-AUC — 간격이 클수록 분포 이동 민감"); axes[0].legend(fontsize=8)
axes[0].grid(axis="y", lw=0.6, alpha=0.5); axes[0].set_axisbelow(True)

if len(roll_df):
    axes[1].plot(roll_df["윈도우 시작"], roll_df["PR-AUC"], "o-", color="#55A868", ms=5)
    axes[1].axhline(_te, color="#DD8452", ls="--", lw=1, label=f"test 전체 {_te:.3f}")
    axes[1].axhline(_va, color="#4C72B0", ls=":", lw=1, label=f"val 전체 {_va:.3f}")
    axes[1].set_xlabel("테스트 윈도우 시작(order_idx)"); axes[1].set_title("윈도우별 롤링 PR-AUC (앙상블)")
    axes[1].legend(fontsize=8); axes[1].grid(lw=0.6, alpha=0.5); axes[1].set_axisbelow(True)
else:
    axes[1].text(0.5, 0.5, "양성 5건 이상 윈도우 없음", ha="center")
plt.tight_layout(); plt.show()

# ── (5) 운영 리포트 저장용 ──
_worst = max([r["상대 하락(%)"] for r in drift_rows]) / 100
DRIFT_REPORT = {
    "metric": "val_to_test_pr_auc_drop",
    "thresholds": {"warn": DRIFT_WARN, "alert": DRIFT_ALERT, "psi_warn": PSI_WARN, "psi_alert": PSI_ALERT},
    "ensemble": {"val_pr_auc": _va, "test_pr_auc": _te, "rel_drop": _rel_ens,
                 "status": drift_status(_rel_ens)},
    "per_seed": [{k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                  for k, v in row.items()} for row in drift_rows[:-1]],
    "score_psi": score_psi, "psi_status": psi_status,
    "alert_rate": {"budget": ALERT_BUDGET, "realized_at_val_thr": realized, "ratio": ratio,
                   "status": rate_status},
    "rolling_pr_auc": roll_df.to_dict("records") if len(roll_df) else [],
    "recommendation": ("재학습 트리거 + 임계값 재캘리브레이션" if _rel_ens >= DRIFT_ALERT else
                       "원인 분석 착수 (피처 드리프트 분해)" if _rel_ens >= DRIFT_WARN else
                       "정상 — 정기 모니터링 유지"),
}
print(f"\n종합 판정: 앙상블 상대 하락 {_rel_ens*100:.1f}% [{drift_status(_rel_ens)}], "
      f"PSI [{psi_status}], 경보율 [{rate_status}]")
print(f"권장 조치: {DRIFT_REPORT['recommendation']}")
print("(§19에서 drift_report.json 으로 저장 — 운영 대시보드/경보 시스템 연동용)")

## 17. 케이스 조사 지원 → STR 초안 (화두 4)

1. 고위험 거래 Top-K 추출 → 2. 주변 계좌 관계 서브그래프 → 3. 조사노트 JSON → 4. STR(의심거래보고) 초안 생성

STR 초안은 **규칙 기반 템플릿**입니다. 실제 STR은 법적 문서라 사람이 검토·서명해야 하고,
자동 생성물을 그대로 제출해서는 안 됩니다.

In [ ]:
TOP_K = 4
os.makedirs("case_notes", exist_ok=True)

test_global_idx = np.arange(n_va, n)
top_local = np.argsort(-ens_t)[:TOP_K]
top_global = test_global_idx[top_local]

fig, axes = plt.subplots(1, TOP_K, figsize=(5.2 * TOP_K, 5))
axes = np.atleast_1d(axes)
notes = []

for ax, gi, li in zip(axes, top_global, top_local):
    row = df.iloc[gi]
    lo = max(0, gi - CONTEXT_EDGES)
    ctx = df.iloc[lo:gi + 1]
    center = {int(row["src_id"]), int(row["dst_id"])}
    nbh = ctx[ctx["src_id"].isin(center) | ctx["dst_id"].isin(center)].tail(60)

    G = nx.MultiDiGraph()
    for _, r in nbh.iterrows():
        G.add_edge(int(r["src_id"]), int(r["dst_id"]), laundering=int(r["Is Laundering"]))
    pos_l = nx.spring_layout(G, seed=SEED, k=1.0)
    nx.draw_networkx_nodes(G, pos_l, ax=ax, node_size=170,
                           node_color=["#C44E52" if v in center else "#A8C6E8" for v in G.nodes()])
    nx.draw_networkx_edges(G, pos_l, ax=ax, arrows=True, alpha=0.7, connectionstyle="arc3,rad=0.06",
                           edge_color=["#C44E52" if a.get("laundering") else "#BBBBBB"
                                       for _, _, a in G.edges(data=True)])
    ax.set_title(f"거래#{gi}  위험도={ens_t[li]:.3f}\n실제라벨={int(row['Is Laundering'])}", fontsize=10)
    ax.axis("off")

    note = {
        "transaction_order_idx": int(gi),
        "risk_score": float(ens_t[li]),
        "threshold_f1": float(row_ens["thr_F1"]),
        "threshold_budget": float(row_ens["thr_예산"]),
        "timestamp": str(row["Timestamp"]),
        "from": f"{row['From Bank']}_{row['Account']}",
        "to": f"{row['To Bank']}_{row['Account.1']}",
        "amount_paid": float(row["Amount Paid"]), "payment_currency": str(row["Payment Currency"]),
        "amount_received": float(row["Amount Received"]), "receiving_currency": str(row["Receiving Currency"]),
        "payment_format": str(row["Payment Format"]),
        "neighborhood_tx_count": int(len(nbh)),
        "neighborhood_laundering_count": int(nbh["Is Laundering"].sum()),
        "signals": {                                   # STR 초안 근거로 쓸 정량 신호
            "pair_first_transaction": bool(row.get("pair_first", 0) == 1),
            "sender_tx_last_1h": float(np.expm1(row.get("out_cnt3600", 0) * np.log(50))),
            "receiver_tx_last_1h": float(np.expm1(row.get("in_cnt3600", 0) * np.log(50))),
            "currency_mismatch": bool(row["cur_mismatch"] == 1),
            "amount_mismatch": bool(row["amt_neq"] == 1),
            "self_loop": bool(row["self_loop"] == 1),
        },
        "investigator_notes": "",
        "linked_str_draft": f"case_notes/str_{gi}.md",
    }
    with open(f"case_notes/case_{gi}.json", "w", encoding="utf-8") as fp:
        json.dump(note, fp, ensure_ascii=False, indent=2)
    notes.append(note)

plt.tight_layout(); plt.show()
print(f"조사노트 {len(notes)}건 저장 -> ./case_notes/")

In [ ]:
def make_str_draft(note: dict) -> str:
    '''조사노트 JSON → STR(의심거래보고) 초안 마크다운. 규칙 기반, 사람 검토 필수.'''
    s = note["signals"]
    grounds = []
    if s["pair_first_transaction"]:
        grounds.append("- 송금인과 수취인 간 **최초 거래**임에도 고액이 이동함 (신규 상대방 리스크)")
    if s["sender_tx_last_1h"] >= 3:
        grounds.append(f"- 송금인이 직전 1시간 내 **{s['sender_tx_last_1h']:.0f}건**을 송금 (분산 송금/팬아웃 의심)")
    if s["receiver_tx_last_1h"] >= 3:
        grounds.append(f"- 수취인이 직전 1시간 내 **{s['receiver_tx_last_1h']:.0f}건**을 수취 (집금/팬인 의심)")
    if s["currency_mismatch"]:
        grounds.append("- 지급 통화와 수취 통화가 상이함 (크로스커런시 레이어링 가능성)")
    if s["amount_mismatch"]:
        grounds.append("- 지급 금액과 수취 금액이 불일치함")
    if s["self_loop"]:
        grounds.append("- 동일 계좌 간 이체(자기송금)로 자금 출처 은폐 가능성")
    if note["neighborhood_laundering_count"] > 0:
        grounds.append(f"- 주변 {note['neighborhood_tx_count']}건 중 "
                       f"**{note['neighborhood_laundering_count']}건**이 기존 의심 거래로 연결됨")
    if not grounds:
        grounds.append("- 개별 규칙 신호는 없으나, 그래프 신경망 모형이 이웃 거래 맥락에서 고위험으로 판정함")

    gi = note["transaction_order_idx"]
    lvl = "높음" if note["risk_score"] >= note["threshold_budget"] else "중간"
    body = chr(10).join(grounds)
    opinion = note["investigator_notes"] or "_(작성 필요)_"
    ts, frm, to = note["timestamp"], note["from"], note["to"]
    score, thb = note["risk_score"], note["threshold_budget"]
    ap, pc = note["amount_paid"], note["payment_currency"]
    ar, rc = note["amount_received"], note["receiving_currency"]
    pfmt, nbc = note["payment_format"], note["neighborhood_tx_count"]
    return f'''# 의심거래보고(STR) 초안 — 거래 #{gi}

> ⚠️ **자동 생성 초안입니다.** 보고 책임자는 반드시 원거래·KYC 자료를 확인하고 본문을 수정·확정해야 합니다.

## 1. 보고 개요
| 항목 | 내용 |
|---|---|
| 탐지 시각(거래 시각) | {ts} |
| 탐지 모형 | 방향 멀티그래프 GNN 엣지 분류 (v7, {len(SEED_LIST)}시드 앙상블 + 계좌 보조 학습) |
| 위험도 점수 | {score:.4f} (경보 예산 임계값 {thb:.4f}) |
| 위험 등급 | {lvl} |

## 2. 거래 내역
| 항목 | 내용 |
|---|---|
| 송금인 (은행_계좌) | {frm} |
| 수취인 (은행_계좌) | {to} |
| 지급 금액 | {ap:,.2f} {pc} |
| 수취 금액 | {ar:,.2f} {rc} |
| 지급 수단 | {pfmt} |

## 3. 의심 근거
{body}

## 4. 주변 거래 관계
직전 컨텍스트 구간에서 해당 계좌들과 연결된 거래 **{nbc}건**을 확인함
(서브그래프는 조사 화면 §17 참조).

## 5. 조사자 의견
{opinion}

## 6. 첨부
- 조사노트 원본: `case_notes/case_{gi}.json`
'''


def refine_with_llm(draft: str) -> str:
    '''LLM 문장 다듬기 슬롯 (MVP 범위 밖 — 네트워크 호출 없음).

    운영 도입 시: 초안을 LLM에 넘겨 문체·용어를 규정 서식에 맞추되,
    **수치·계좌번호는 원본에서 그대로 가져오도록** 제약을 걸어야 합니다(환각 방지).
    '''
    return draft


for note in notes:
    p = Path(note["linked_str_draft"])
    p.write_text(refine_with_llm(make_str_draft(note)), encoding="utf-8")
print(f"STR 초안 {len(notes)}건 저장 -> ./case_notes/str_*.md\n")
print(Path(notes[0]["linked_str_draft"]).read_text(encoding="utf-8")[:1400])

## 18. 분 단위 배치 추론 데모 (화두 2)

학습 윈도우와 동일한 구조(과거 `CONTEXT_EDGES` 건 + 대상 구간)로 채점합니다 — 학습·서빙 그래프 정합 유지.

In [ ]:
@torch.no_grad()
def score_batch(batch_end_gi, batch_size, models=None):
    models = models or ens_models
    lo = batch_end_gi + 1 - batch_size
    g = to_dev(build_window(max(0, lo - CONTEXT_EDGES), lo, batch_end_gi + 1))
    ps = []
    for mdl in models:
        mdl.eval()
        with torch.autocast("cuda", torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
            el, _ = mdl(g)
        ps.append(torch.sigmoid(el.float()).cpu().numpy())
    return np.mean(ps, axis=0)

test_df = df.iloc[n_va:]
minute_groups = test_df.groupby(test_df["Timestamp"].dt.floor("min"), sort=True)

N_DEMO = 30
lat_ens, lat_one, sizes = [], [], []
for i, (minute, g_) in enumerate(minute_groups):
    if i >= N_DEMO:
        break
    end_gi = int(g_["order_idx"].iloc[-1])
    t = time.perf_counter(); score_batch(end_gi, len(g_)); lat_ens.append((time.perf_counter()-t)*1000)
    t = time.perf_counter(); score_batch(end_gi, len(g_), ens_models[:1]); lat_one.append((time.perf_counter()-t)*1000)
    sizes.append(len(g_))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(np.arange(len(lat_ens))-0.2, lat_ens, width=0.4, color="#4C72B0", label=f"앙상블({len(ens_models)})")
axes[0].bar(np.arange(len(lat_one))+0.2, lat_one, width=0.4, color="#55A868", label="단일 모델")
axes[0].set_xlabel("분 단위 배치 #"); axes[0].set_ylabel("지연(ms)"); axes[0].legend(fontsize=8)
axes[0].set_title(f"배치별 추론 지연 (중앙값 앙상블 {np.median(lat_ens):.0f} / 단일 {np.median(lat_one):.0f} ms)")
axes[1].scatter(sizes, lat_ens, color="#C44E52", alpha=0.7)
axes[1].set_xlabel("배치 크기(거래 수)"); axes[1].set_ylabel("지연(ms)")
axes[1].set_title("배치 크기 vs 지연 — 화두2 위험(급증 시 지연) 확인용")
plt.tight_layout(); plt.show()
print(f"앙상블은 단일 대비 지연 {np.median(lat_ens)/max(np.median(lat_one),1e-9):.1f}배 — "
      f"SLA가 빠듯하면 §12 표의 '단일 평균' 성능으로 판단하세요.")

## 19. 모델·설정·리포트 저장 (배포/재현용)

[v7] 스윕 결과·A/B 결과·전이 비교표·**드리프트 리포트**를 함께 저장합니다 — 운영 대시보드가
`drift_report.json` 을 주기적으로 읽어 경보를 울리는 구성을 가정합니다.

In [ ]:
os.makedirs("artifacts_v7", exist_ok=True)
for r in runs:
    torch.save(r["state"], f"artifacts_v7/{DATASET}_edge_gine_seed{r['seed']}.pt")

def _num(o):
    '''np 스칼라 → 파이썬 기본형 (json 직렬화용).'''
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    raise TypeError(str(type(o)))

with open(f"artifacts_v7/{DATASET}_config.json", "w", encoding="utf-8") as fp:
    json.dump({
        "version": "v7-generalization", "dataset": DATASET, "seeds": SEED_LIST,
        "edge_features": EDGE_FEATURES, "time_features": TIME_FEATURES,
        "node_dim": NODE_FEAT_DIM, "edge_dim": EDGE_DIM, "edge_extra": EDGE_EXTRA,
        "hidden": HIDDEN, "layers": N_LAYERS, "dropout": DROPOUT,
        "use_time_feats": USE_TIME_FEATS, "use_ports": USE_PORTS, "use_mega": USE_MEGA,
        "window_size": WINDOW_SIZE, "context_edges": CONTEXT_EDGES,
        "pos_weight_cap": POS_WEIGHT_CAP,
        "pos_cap_sweep": [{"cap": r_["cap"], "val_pr_auc": r_["val PR-AUC"],
                           "test_pr_auc": r_["test PR-AUC"]} for r_ in POS_SWEEP_ROWS],
        "aux_node": {"enabled": USE_NODE_AUX, "weight": AUX_NODE_W},
        "alert_budget": ALERT_BUDGET,
        "threshold_f1": float(row_ens["thr_F1"]), "threshold_budget": float(row_ens["thr_예산"]),
        "val_pr_auc_per_seed": [r["val_ap"] for r in runs],
        "test_pr_auc_ensemble": float(row_ens["test PR-AUC"]),
        "test_pr_auc_single_mean": float(res["test PR-AUC"].mean()),
        "test_pr_auc_single_std": float(res["test PR-AUC"].std(ddof=1)),
        "account_level_eval": ACCT_RESULT,
        "time_feature_ab": TIME_AB_RESULT,
        "transfer_eval": TRANSFER_ROWS,
        "pf_cats": PF_CATS, "pc_cats": PC_CATS, "rc_cats": RC_CATS,
        "tail_min_frac": TAIL_MIN_FRAC,
        "main_period": [str(main_start.date()), str(main_end.date())],
    }, fp, ensure_ascii=False, indent=2, default=_num)

with open(f"artifacts_v7/{DATASET}_drift_report.json", "w", encoding="utf-8") as fp:
    json.dump(DRIFT_REPORT, fp, ensure_ascii=False, indent=2, default=_num)

res.to_csv(f"artifacts_v7/{DATASET}_seed_results.csv", index=False)
print("저장 완료 -> ./artifacts_v7/  (config / drift_report / seed_results / 모델 4개)")

## 20. 결론 및 다음 단계

### v7이 한 일 — v6 §19 남은 과제 5개 전부 반영

| # | 과제 | 구현 | 산출물 |
|---|---|---|---|
| 1 | 세트 간 비교 (화두 7) | §15 zero-shot 전이 평가 — HI-Small 앙상블을 타 세트에 무수정 적용, lift로 유병률 보정 비교. 파이프라인 함수화로 파일만 오면 재학습 없이 즉시 실행 | `transfer_eval` (config.json) |
| 2 | POS_WEIGHT_CAP 스윕 | §10-2 — 5/15/30 스윕, **검증 PR-AUC로만 선택**해 본학습·GBT에 반영 | `pos_cap_sweep` |
| 3 | 계좌 단위 보조 판정 | §9 계좌 보조 헤드 + §10 멀티태스크 손실(학습 정규화) + §12-2 계좌 집계 평가(최댓값/상위3) | `account_level_eval` |
| 4 | 드리프트 모니터링 | §16 — 상대 하락 주의/경보 임계값, PSI(라벨 불필요 조기 경보), 경보율 안정성, 롤링 PR-AUC | `drift_report.json` |
| 5 | 시간차 제거 A/B | §13-2 — 재학습 기반 A/B + 하락폭별 이관 조치 기준표 | `time_feature_ab` |

### 실데이터 이관 체크리스트 (v7 산출물 기준)
1. §13-2 판정이 "중간 의존" 이상이면 → 실데이터에서 시간차 피처 포함/제거 챔피언-챌린저 재실험
2. §15 zero-shot lift 유지율이 낮으면 → 학습 세트 특이 패턴 의존 의심, 실데이터 재학습 필수 전제
3. `drift_report.json` 의 PSI·경보율 지표를 운영 파이프라인에 주간 배치로 편입 (라벨 도착 전 조기 경보)
4. STR 초안은 규칙 기반 유지 — LLM 연계 시 수치·계좌번호 불변 제약 필수

### 남은 것 / 권장 다음 단계
* **세트별 전면 재학습 비교** — zero-shot(§15)은 전이력 측정. 세트별 최적 성능 비교는 `DATASET` 변경 후
  전체 재실행이 필요 (Medium 세트는 메모리 프로파일링 선행)
* **계좌 보조 판정 고도화** — 현재는 윈도우-로컬 보조 라벨. 계좌 단위 시퀀스 모델(거래 이력 인코더) 병행 검토
* **서브그래프(시나리오) 단위 평가** — `*_Patterns.txt` 의 시나리오 블록 재현율(패턴 중 몇 %를 최소 1건이라도
  탐지했는가)을 §12에 추가하면 화두 1을 평가 지표 차원에서도 닫을 수 있음
* **하이퍼파라미터 확장 스윕** — cap 외에 `HIDDEN`/`N_LAYERS`/`AUX_NODE_W` 도 §10-2 틀로 확장 가능